# 03 Pilot extraction: six reports, the record schema, one revision, freeze

Design log.
- Plan v2 (B3) makes notebook 03 a pilot: write the record schema and the coding rules, fill records for six varied reports read in full, list every place where the schema needed a free-text workaround, revise the schema once and freeze it. Amendment 1 adds two training-target fields to every defence (whether training pairs the user's own instruction with the same injected words, and whether it trains instruction-bearing text to be carried through), taken from the last two columns of Table 1 of the twin distillation paper.
- The six reports, one per slot named in the plan: a template defence, Spotlighting (2403.14720); a training defence, SecAlign (2410.05451, CCS 2025); an embedding defence, ASIDE (2503.10566, ICLR 2026); an internal intervention, CounterSteer (2609.36570); an outside re-measurement, Jia et al.'s critical evaluation (2505.18333); a fidelity study, SecFid (2606.30783). Each was read from its PDF in data/defences; Stage 2 checks that each PDF is the one registered in reports/defences.csv.
- Records follow plan v2 A6: report, defences, evaluations, results, comparisons, claims and coverage. Every value is copied as printed with its page; every claim and the key defence facts carry a short verbatim anchor. Stage 3 finds every value and every anchor on its page by code. Comparisons are graded A, B, C or U by plan v2 equation (4) in code, not by hand.
- The pilot was coded first with schema v0. The 33 places where v0 needed a workaround are kept in reports/pilot_v0/ (Stage 4 lists them). Schema v1 answers them in 22 numbered changes (schema/pilot_changes.md): 20 change the schema, which Stage 5 counts from the two files as new fields and widened value sets, and 2 are coding rules. The six reports were recoded with v1 and v1 is frozen here. Two notes stay as text: one paper-specific reading aid and one reading problem.
- What the pilot showed the schema must capture (these are reading aids for notebook 05, not findings): a paper can print one defence name and run another checkpoint (SecFid prints SecAlign and runs facebook/Meta-SecAlign-8B); a utility claim can rest on general benchmarks while the paper's own closer measure moves the other way (ASIDE's SEP Utility); the same released checkpoint can be re-measured with a different reference and judge (Jia et al. on SecAlign); utility can be reported relative to a reference (SEP utility, percentage of a row's own clean run).
- The checker also changed during the pilot: values are looked for in a column-preserving extraction as well as the plain text, because the plain text glues table cells together; YAML is read so that yes and no stay words; anchors are at most 14 words.
- Outputs: src/schema.py, tests/test_schema.py, schema/record_schema.yaml (v1, frozen), schema/record_schema_v0.yaml, schema/coding_rules.md, schema/pilot_changes.md, reports/records/ (six files), reports/pilot_v0/ (six files), reports/pilot_03_check.csv, reports/pilot_03_gaps.csv, reports/pilot_03_summary.csv, reports/decision_03.txt.
- CPU; no network calls; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import pypdf
if int(pypdf.__version__.split(".")[0]) < 4:                 # the column-preserving extraction needs pypdf 4 or later
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pypdf>=4"], check=False)
    print("pypdf upgraded; restart the runtime if the next stage reports an old version")
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd(), "| pypdf", pypdf.__version__)

## Stage 1. Schema module, tests, the frozen schema, coding rules, the pilot records

In [ ]:
Path("reports/records").mkdir(parents=True, exist_ok=True)       # %%writefile does not create folders
Path("reports/pilot_v0").mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile src/schema.py
"""Record schema: load it, validate record files against it and against the PDFs they cite, grade comparisons."""
from __future__ import annotations

import re
import unicodedata
from pathlib import Path

import yaml

CONDITIONS = ["c1_base", "c2_layout", "c3_benchmark", "c4_attacker", "c5_judge", "c6_decoding"]


class _Loader(yaml.SafeLoader):
    """YAML loader that reads only true/false as booleans, so the enum words yes and no stay words (YAML 1.1 would
    silently turn yes, no, on and off into booleans)."""


_Loader.yaml_implicit_resolvers = {k: [r for r in v if r[0] != "tag:yaml.org,2002:bool"]
                                   for k, v in yaml.SafeLoader.yaml_implicit_resolvers.items()}
_Loader.add_implicit_resolver("tag:yaml.org,2002:bool", re.compile(r"^(?:true|True|TRUE|false|False|FALSE)$"),
                              list("tTfF"))


def load_yaml(path: Path):
    return yaml.load(Path(path).read_text(encoding="utf-8"), Loader=_Loader)


def load_schema(path: Path) -> dict:
    return load_yaml(path)


def load_record(path: Path) -> dict:
    return load_yaml(path) or {}


def norm_text(s: str) -> str:
    """Lower-case alphanumerics only (the same rule as the overlap and full-text anchor checks)."""
    s = unicodedata.normalize("NFKC", s or "").replace("­", "")
    return re.sub(r"[^a-z0-9]+", "", s.lower())


def norm_value(s: str) -> str:
    """A printed number as it can be searched for: NFKC, unicode minus as hyphen, runs of whitespace as one space."""
    s = unicodedata.normalize("NFKC", str(s)).replace("\u2212", "-").replace("\u2013", "-")
    return re.sub(r"\s+", " ", s).strip()


def value_on_page(value: str, page_text: str) -> bool:
    """True when the printed value appears on the page as a number of its own (not inside a longer number).
    A space before % is allowed (3.10% matches 3.10 %); a trailing % may sit in a table header instead."""
    v = norm_value(value).replace(" ", "")
    t = norm_value(page_text)
    if not v:
        return False
    pat = re.escape(v).replace("%", r"\s?%")
    if re.search(rf"(?<![\d.]){pat}(?![\d])", t):
        return True
    if v.endswith("%"):
        core = re.escape(v[:-1])
        return bool(re.search(rf"(?<![\d.]){core}(?![\d])", t))
    return False


def grade(comp: dict) -> str:
    """Plan v2 equation (4): A if every condition is matched or is the one declared manipulated variable; U if any
    condition is unreported; B if exactly one is unmatched; C if two or more are unmatched. For a whole-configuration
    (W) comparison the layout may differ by design and is left out of the grade. "matched_unstated" (both arms ran in
    one harness but the value is not printed) counts as matched inside one report and as unreported in a cross-report
    (X) comparison, where the reader cannot know it was the same."""
    conds = [c for c in CONDITIONS if not (comp.get("purpose") == "W" and c == "c2_layout")]
    vals = [comp.get(c) for c in conds]
    if comp.get("purpose") == "X":
        vals = ["unreported" if v == "matched_unstated" else v for v in vals]
    if "unreported" in vals:
        return "U"
    if vals.count("manipulated") > 1:
        return "C"
    n = vals.count("unmatched")
    return "A" if n == 0 else "B" if n == 1 else "C"


def _check_field(kind: str, rid: str, name: str, spec: dict, val, out: list[str]) -> None:
    t = spec.get("type")
    where = f"{kind}:{rid}:{name}"
    if t == "enum" and val not in spec["values"]:
        out.append(f"{where}: {val!r} is not one of {spec['values']}")
    elif t == "enum_list":
        if not isinstance(val, list) or not val:
            out.append(f"{where}: needs a non-empty list")
        else:
            out += [f"{where}: {v!r} is not one of {spec['values']}" for v in val if v not in spec["values"]]
    elif t == "map":
        if not isinstance(val, dict):
            out.append(f"{where}: needs a mapping")
        else:
            for k, v in val.items():
                if k not in spec["keys"]:
                    out.append(f"{where}: key {k!r} is not one of {spec['keys']}")
                elif v not in spec["values"]:
                    out.append(f"{where}.{k}: {v!r} is not one of {spec['values']}")
    elif t == "pages":
        if not isinstance(val, dict) or not all(isinstance(v, int) and v >= 1 for v in val.values()):
            out.append(f"{where}: needs a mapping of field names to positive page numbers")
    elif t == "map_free" and not isinstance(val, dict):
        out.append(f"{where}: needs a mapping")
    elif t == "page" and not (isinstance(val, int) and val >= 1):
        out.append(f"{where}: page must be a positive integer")
    elif t == "int" and not isinstance(val, int):
        out.append(f"{where}: must be an integer")
    elif t == "list" and not isinstance(val, list):
        out.append(f"{where}: must be a list")
    elif t == "str" and not isinstance(val, (str, int, float)):
        out.append(f"{where}: must be text")


def validate_record(rec: dict, schema: dict, page_texts: list[str] | None = None,
                    page_layouts: list[str] | None = None) -> tuple[list[str], list[str]]:
    """Return (problems, notes). Problems break the schema or fail a check against the PDF; notes are free-text
    workarounds the coder recorded in schema_gaps. page_texts are the plain text of each PDF page (anchors are found
    there); page_layouts, when given, are the same pages extracted with column spacing kept, so that numbers in table
    rows are not glued together (values are looked for in either)."""
    problems: list[str] = []
    kinds = schema["kinds"]
    for k in rec:
        if k not in kinds:
            problems.append(f"unknown record kind: {k}")
    ids: dict[str, set] = {}
    for kind, kspec in kinds.items():
        fields = kspec["fields"]
        items = rec.get(kind)
        if items is None:
            if kind in ("report", "coverage", "defences"):
                problems.append(f"missing record kind: {kind}")
            continue
        rows = [items] if kspec.get("one") else items
        if not isinstance(rows, list):
            problems.append(f"{kind}: must be a list")
            continue
        seen = ids.setdefault(kind, set())
        for row in rows:
            if not isinstance(row, dict):
                problems.append(f"{kind}: every entry must be a mapping")
                continue
            rid = str(row.get("id", kind))
            if "id" in fields and rid in seen:
                problems.append(f"{kind}:{rid}: duplicate id")
            seen.add(rid)
            for name in row:
                if name not in fields:
                    problems.append(f"{kind}:{rid}: unknown field {name!r}")
            for name, spec in fields.items():
                if name not in row or row[name] in (None, ""):
                    if spec.get("required"):
                        problems.append(f"{kind}:{rid}: missing required field {name!r}")
                    continue
                _check_field(kind, rid, name, spec, row[name], problems)
            needs_page = any(spec.get("from_paper") and name in row for name, spec in fields.items())
            page = row.get("page")
            if needs_page and "page" in fields and not isinstance(page, int):
                problems.append(f"{kind}:{rid}: fields read from the paper need a page")
            if page_texts is not None and isinstance(page, int):
                if page > len(page_texts):
                    problems.append(f"{kind}:{rid}: page {page} is beyond the PDF's {len(page_texts)} pages")
                else:
                    text = page_texts[page - 1]
                    lay = page_layouts[page - 1] if page_layouts else ""
                    if kind == "results" and row.get("value_source") != "figure_only":
                        if row.get("value") in (None, ""):
                            problems.append(f"{kind}:{rid}: a printed result needs its value")
                        elif not (value_on_page(row["value"], text) or value_on_page(row["value"], lay)):
                            problems.append(f"{kind}:{rid}: value {row['value']!r} not found on page {page}")
                    if row.get("anchor") and len(str(row["anchor"]).split()) > 14:
                        problems.append(f"{kind}:{rid}: anchor is longer than 14 words; keep it a short phrase")
                    if row.get("anchor") and norm_text(row["anchor"]) not in norm_text(text):
                        nxt = page_texts[page] if page < len(page_texts) else ""
                        if norm_text(row["anchor"]) not in norm_text(text + nxt):
                            problems.append(f"{kind}:{rid}: anchor not found on page {page}")
            if page_texts is not None and isinstance(row.get("pages"), dict):
                for fld, pg in row["pages"].items():
                    if isinstance(pg, int) and pg > len(page_texts):
                        problems.append(f"{kind}:{rid}: pages.{fld} = {pg} is beyond the PDF's {len(page_texts)} pages")
    rep_id = (rec.get("report") or {}).get("id")
    defence_ids = ids.get("defences", set())
    eval_ids = ids.get("evaluations", set())
    for ev in rec.get("evaluations") or []:
        d = str(ev.get("defence_id", ""))
        if not d.startswith("none:") and d not in defence_ids and ":" not in d:
            problems.append(f"evaluations:{ev.get('id')}: defence_id {d!r} is not a defence in this file")
        mb = ev.get("measured_by")
        if mb and mb != rep_id and not str(mb).endswith("_ext"):
            pass                                            # another report's id: checked across files in notebook 04
    for r in rec.get("results") or []:
        if r.get("evaluation_id") not in eval_ids:
            problems.append(f"results:{r.get('id')}: evaluation_id {r.get('evaluation_id')!r} is not in this file")
    comp_ids = ids.get("comparisons", set())
    for c in rec.get("comparisons") or []:
        for side in ("left", "right"):
            if c.get(side) not in eval_ids:
                problems.append(f"comparisons:{c.get('id')}: {side} {c.get(side)!r} is not an evaluation in this file")
        if c.get("purpose") == "M" and [c.get(x) for x in CONDITIONS].count("manipulated") > 1:
            problems.append(f"comparisons:{c.get('id')}: a mechanism comparison can manipulate at most one condition")
    for k in rec.get("claims") or []:
        for s in k.get("supported_by") or []:
            if s not in comp_ids:
                problems.append(f"claims:{k.get('id')}: supported_by {s!r} is not a comparison in this file")
    res_ids = ids.get("results", set())
    for k in rec.get("claims") or []:
        for r_ in k.get("contradicted_by") or []:
            if r_ not in res_ids:
                problems.append(f"claims:{k.get('id')}: contradicted_by {r_!r} is not a result in this file")
    for ev in rec.get("evaluations") or []:
        sc = ev.get("same_checkpoint_as")
        if sc and ":" not in str(sc):
            problems.append(f"evaluations:{ev.get('id')}: same_checkpoint_as must read <report_id>:<defence_id>")
    notes = [f"{g.get('where')}: {g.get('note')}" for g in rec.get("schema_gaps") or []]
    return problems, notes

In [ ]:
%%writefile tests/test_schema.py
"""The record schema, the validator and the comparison grade."""
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
import schema  # noqa: E402

SCHEMA = schema.load_schema(ROOT / "schema" / "record_schema.yaml")


def comp(purpose="W", **kw):
    c = {"purpose": purpose, "c1_base": "matched", "c2_layout": "matched", "c3_benchmark": "matched",
         "c4_attacker": "matched", "c5_judge": "matched", "c6_decoding": "matched"}
    c.update(kw)
    return c


def test_grade_follows_plan_equation_4():
    assert schema.grade(comp()) == "A"
    assert schema.grade(comp(c5_judge="unmatched")) == "B"
    assert schema.grade(comp(c1_base="unmatched", c3_benchmark="unmatched")) == "C"
    assert schema.grade(comp(c6_decoding="unreported")) == "U"
    assert schema.grade(comp("M", c2_layout="manipulated")) == "A"
    assert schema.grade(comp("M", c2_layout="manipulated", c4_attacker="manipulated")) == "C"


def test_whole_configuration_comparison_ignores_layout():
    assert schema.grade(comp("W", c2_layout="unmatched")) == "A"
    assert schema.grade(comp("M", c2_layout="unmatched")) == "B"


def test_matched_unstated_is_matched_within_a_report_and_unknown_across_reports():
    assert schema.grade(comp("W", c6_decoding="matched_unstated")) == "A"
    assert schema.grade(comp("X", c6_decoding="matched_unstated")) == "U"


def test_value_search_keeps_table_cells_apart():
    assert schema.value_on_page("59", "None 59 2 0")
    assert schema.value_on_page("3.10%", "reduced to 3.10 % with")
    assert schema.value_on_page("13.6", "Vanilla 13.6±0.2 17.9")
    assert not schema.value_on_page("65", "cells 6 5 here")
    assert not schema.value_on_page("1", "only 11 and 0.1")
    assert not schema.value_on_page("0.5", "10.5 and 0.55")


def test_loader_keeps_yes_and_no_as_words(tmp_path):
    f = tmp_path / "r.yaml"
    f.write_text("a: yes\nb: no\nc: true\n")
    d = schema.load_record(f)
    assert d == {"a": "yes", "b": "no", "c": True}


def test_schema_flags_are_booleans_and_enum_words_are_text():
    for kind in SCHEMA["kinds"].values():
        for spec in kind["fields"].values():
            for flag in ("required", "from_paper"):
                if flag in spec:
                    assert isinstance(spec[flag], bool)
            for v in spec.get("values", []):
                assert isinstance(v, str)


def minimal_record():
    return {
        "report": {"id": "r", "slug": "2401.00001", "title": "t", "first_author": "a", "venue": "v", "peer_reviewed": "no",
                   "version_read": "arXiv v1", "first_date": "2024-01-01", "read": "F", "role": "proposes",
                   "extraction_scope": "whole report"},
        "defences": [{"id": "d", "name": "D", "origin": "this_report", "base_checkpoint": "M", "axis1": ["chat_roles"],
                      "axis2": ["sft"], "isolation": "no_ablation", "signal_dependence": "not_reported",
                      "trains_instruction_pair": "no", "trains_carry_through": "no", "page": 2}],
        "evaluations": [{"id": "e", "defence_id": "d", "benchmark": "B", "attacker_type": "static", "attacker_access": "black_box",
                         "judge": "j", "layout_same_as_defence": "yes", "measured_by": "r", "page": 3},
                        {"id": "e0", "defence_id": "none:M", "benchmark": "B", "attacker_type": "static", "attacker_access": "black_box",
                         "judge": "j", "layout_same_as_defence": "yes", "measured_by": "r", "page": 3}],
        "results": [{"id": "x", "evaluation_id": "e", "quantity": "R1", "coverage": "direct", "metric_as_printed": "ASR",
                     "value": "4.5", "unit": "percent", "direction": "lower_better", "qualifier": "exact", "value_source": "table", "page": 3},
                    {"id": "y", "evaluation_id": "e", "quantity": "R2clean", "coverage": "proxy", "metric_as_printed": "chart",
                     "unit": "percent", "direction": "higher_better", "qualifier": "approximate", "value_source": "figure_only", "page": 3}],
        "comparisons": [dict(comp(), id="c", left="e", right="e0", varies="the defence", page=3)],
        "claims": [{"id": "k", "text": "t", "location": "abstract", "anchor": "a short phrase", "jobs": ["R1"],
                    "supported_by": ["c"], "contradicted_by": ["y"], "page": 1}],
        "coverage": {"R1": "direct", "R2clean": "proxy", "R2attack": "not_reported", "R3ns": "not_reported",
                     "R3cp": "not_reported", "R3safe": "not_reported"},
    }


def test_a_valid_record_passes_with_its_pages():
    pages = ["intro", "method", "Table 2: ASR 4.5 under attack", "x"]
    pages[0] = "a short phrase in the abstract"
    probs, notes = schema.validate_record(minimal_record(), SCHEMA, pages)
    assert probs == [] and notes == []


def test_validator_reports_each_kind_of_fault():
    rec = minimal_record()
    rec["defences"][0]["axis1"] = ["telepathy"]
    rec["results"][0]["value"] = "9.9"
    rec["results"][0]["page"] = 9
    rec["comparisons"][0]["right"] = "nowhere"
    rec["claims"][0]["anchor"] = "one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen"
    del rec["report"]["extraction_scope"]
    probs, _ = schema.validate_record(rec, SCHEMA, ["one two", "b", "c", "d"])
    text = "\n".join(probs)
    assert "telepathy" in text
    assert "page 9 is beyond" in text
    assert "right 'nowhere'" in text
    assert "longer than 14 words" in text
    assert "extraction_scope" in text


def test_a_printed_result_needs_its_value_but_a_figure_only_one_does_not():
    rec = minimal_record()
    del rec["results"][0]["value"]
    probs, _ = schema.validate_record(rec, SCHEMA, ["a short phrase", "b", "Table 2", "d"])
    assert any("needs its value" in p for p in probs)
    assert not any(":y:" in p for p in probs)


def test_the_pilot_records_follow_the_frozen_schema():
    files = sorted((ROOT / "reports" / "records").glob("*.yaml"))
    assert len(files) >= 6
    for f in files:
        probs, _ = schema.validate_record(schema.load_record(f), SCHEMA)
        assert probs == [], (f.name, probs[:3])

Schema v1, frozen after the pilot.

In [ ]:
%%writefile schema/record_schema.yaml
# Record schema v1, frozen in notebook 03 after the six-report pilot.
# One YAML file per report in reports/records/<report_id>.yaml. How to choose each value: schema/coding_rules.md.
# What changed from the pilot draft (schema/record_schema_v0.yaml) and why: schema/pilot_changes.md.
# Field spec: type, allowed values, required, from_paper (read from the paper, so the record needs a page).
version: 1
page_rule: 'Every record with from_paper fields carries page (1-based PDF page index, within the PDF''s length). A fact on another page goes in pages: {field: page}.'
kinds:
  report:
    one: true
    fields:
      id:
        type: str
        required: true
      slug:
        type: str
        required: true
      title:
        type: str
        required: true
      first_author:
        type: str
        required: true
      venue:
        type: str
        required: true
      peer_reviewed:
        type: enum
        values:
        - 'yes'
        - 'no'
        - unknown
        required: true
      doi:
        type: str
      arxiv:
        type: str
      version_read:
        type: str
        required: true
      first_date:
        type: str
        required: true
      read:
        type: enum
        values:
        - F
        - A
        required: true
      role:
        type: enum
        values:
        - proposes
        - re_measures
        - benchmark
        - mixed
        required: true
      code_release:
        type: str
      later_version:
        type: str
      extraction_scope:
        type: str
        required: true
  defences:
    fields:
      id:
        type: str
        required: true
      name:
        type: str
        required: true
      origin:
        type: enum
        values:
        - this_report
        - other_report
        required: true
      base_checkpoint:
        type: str
        required: true
        from_paper: true
      axis1:
        type: enum_list
        values:
        - chat_roles
        - delimiters_markers
        - encodings
        - learned_token_features
        - fixed_embedding_transform
        - position
        - internal_intervention
        - no_signal
        required: true
      axis2:
        type: enum_list
        values:
        - prompting
        - soft_tokens_frozen
        - sft
        - preference_optimisation
        - distillation
        - adversarial_rl
        - fixed_inference_rule
        required: true
      components:
        type: list
      isolation:
        type: enum
        values:
        - ablation_present
        - no_ablation
        - not_applicable
        required: true
      layout:
        type: str
        from_paper: true
      signal_dependence:
        type: enum
        values:
        - measured
        - measured_other_checkpoint
        - stated
        - baseline_only
        - not_reported
        - not_applicable
        required: true
      layout_compat:
        type: map
        keys:
        - user_turn
        - separate_role
        - system_context
        - agent_tool_result
        values:
        - supported
        - unsupported
        - untested
      released_weights:
        type: str
      trains_instruction_pair:
        type: enum
        values:
        - 'yes'
        - partly
        - 'no'
        - not_applicable
        required: true
      trains_carry_through:
        type: enum
        values:
        - yes_exact_targets
        - yes_by_preference
        - 'no'
        - not_applicable
        required: true
      page:
        type: page
      anchor:
        type: str
      name_as_printed:
        type: str
      checkpoint_id:
        type: str
      signal_dependence_detail:
        type: str
      pages:
        type: pages
  evaluations:
    fields:
      id:
        type: str
        required: true
      defence_id:
        type: str
        required: true
      benchmark:
        type: str
        required: true
        from_paper: true
      split:
        type: str
      attacker_type:
        type: enum
        values:
        - static
        - adaptive
        - optimisation
        - human
        - rl
        - none
        required: true
      attacker_access:
        type: enum
        values:
        - black_box
        - grey_box
        - white_box
        - not_applicable
        - unreported
        required: true
      attacker_budget:
        type: str
      judge:
        type: str
        required: true
      denominator:
        type: str
      decoding:
        type: str
      sample_size:
        type: str
      layout_same_as_defence:
        type: enum
        values:
        - 'yes'
        - 'no'
        - unclear
        required: true
      measured_by:
        type: str
        required: true
      page:
        type: page
      baseline_kind:
        type: enum
        values:
        - off_the_shelf
        - same_data_twin
        - instruction_prompt_only
        - not_applicable
      run:
        type: str
      injection_position:
        type: enum
        values:
        - start
        - middle
        - end
        - mixed
        - varied
        - unreported
      judge_reading:
        type: enum
        values:
        - single
        - strict
        - relaxed
      subset_of:
        type: str
      locked_split:
        type: enum
        values:
        - 'yes'
        - 'no'
        - unreported
      same_checkpoint_as:
        type: str
      pages:
        type: pages
  results:
    fields:
      id:
        type: str
        required: true
      evaluation_id:
        type: str
        required: true
      quantity:
        type: enum
        values:
        - R1
        - R2clean
        - R2attack
        - R3ns
        - R3cp
        - R3safe
        - general_utility
        - other
        required: true
      coverage:
        type: enum
        values:
        - direct
        - proxy
        required: true
      metric_as_printed:
        type: str
        required: true
        from_paper: true
      value:
        type: str
        required: false
        from_paper: true
      unit:
        type: enum
        values:
        - percent
        - fraction
        - score
        - count
        - percent_of_reference
        required: true
      direction:
        type: enum
        values:
        - higher_better
        - lower_better
        required: true
      location:
        type: str
      page:
        type: page
      also_quantity:
        type: enum_list
        values:
        - R1
        - R2clean
        - R2attack
        - R3ns
        - R3cp
        - R3safe
        - general_utility
        - other
      qualifier:
        type: enum
        values:
        - exact
        - approximate
        - upper_bound
        - lower_bound
        required: true
      value_source:
        type: enum
        values:
        - text
        - table
        - figure_caption
        - figure_only
        required: true
      reference:
        type: str
      spread:
        type: str
  comparisons:
    fields:
      id:
        type: str
        required: true
      left:
        type: str
        required: true
      right:
        type: str
        required: true
      purpose:
        type: enum
        values:
        - W
        - M
        - X
        required: true
      c1_base:
        type: enum
        values:
        - matched
        - matched_unstated
        - unmatched
        - unreported
        - manipulated
        required: true
      c2_layout:
        type: enum
        values:
        - matched
        - matched_unstated
        - unmatched
        - unreported
        - manipulated
        required: true
      c3_benchmark:
        type: enum
        values:
        - matched
        - matched_unstated
        - unmatched
        - unreported
        - manipulated
        required: true
      c4_attacker:
        type: enum
        values:
        - matched
        - matched_unstated
        - unmatched
        - unreported
        - manipulated
        required: true
      c5_judge:
        type: enum
        values:
        - matched
        - matched_unstated
        - unmatched
        - unreported
        - manipulated
        required: true
      c6_decoding:
        type: enum
        values:
        - matched
        - matched_unstated
        - unmatched
        - unreported
        - manipulated
        required: true
      page:
        type: page
      varies:
        type: str
        required: true
  claims:
    fields:
      id:
        type: str
        required: true
      text:
        type: str
        required: true
      location:
        type: str
        required: true
      anchor:
        type: str
        required: true
        from_paper: true
      jobs:
        type: enum_list
        values:
        - R1
        - R2
        - R3
        - general_utility
        required: true
      supported_by:
        type: list
      page:
        type: page
      contradicted_by:
        type: list
  coverage:
    one: true
    fields:
      R1:
        type: enum
        values:
        - direct
        - proxy
        - aggregate_only
        - not_reported
        - unclear
        - not_applicable
        required: true
      R2clean:
        type: enum
        values:
        - direct
        - proxy
        - aggregate_only
        - not_reported
        - unclear
        - not_applicable
        required: true
      R2attack:
        type: enum
        values:
        - direct
        - proxy
        - aggregate_only
        - not_reported
        - unclear
        - not_applicable
        required: true
      R3ns:
        type: enum
        values:
        - direct
        - proxy
        - aggregate_only
        - not_reported
        - unclear
        - not_applicable
        required: true
      R3cp:
        type: enum
        values:
        - direct
        - proxy
        - aggregate_only
        - not_reported
        - unclear
        - not_applicable
        required: true
      R3safe:
        type: enum
        values:
        - direct
        - proxy
        - aggregate_only
        - not_reported
        - unclear
        - not_applicable
        required: true
      where:
        type: map_free
  schema_gaps:
    fields:
      where:
        type: str
        required: true
      note:
        type: str
        required: true

Schema v0, the pilot draft, kept so that the pilot and its gaps can be checked.

In [ ]:
%%writefile schema/record_schema_v0.yaml
# Record schema v0: the pilot draft used in notebook 03 to code the six pilot reports (reports/pilot_v0/).
# Superseded by schema/record_schema.yaml (v1); kept so the pilot and its schema gaps can be checked.
version: 0
page_rule: Every record with from_paper fields carries page (1-based PDF page index, within the PDF's length).
kinds:
  report:
    one: true
    fields:
      id:
        type: str
        required: true
      slug:
        type: str
        required: true
      title:
        type: str
        required: true
      first_author:
        type: str
        required: true
      venue:
        type: str
        required: true
      peer_reviewed:
        type: enum
        values:
        - 'yes'
        - 'no'
        - unknown
        required: true
      doi:
        type: str
      arxiv:
        type: str
      version_read:
        type: str
        required: true
      first_date:
        type: str
        required: true
      read:
        type: enum
        values:
        - F
        - A
        required: true
      role:
        type: enum
        values:
        - proposes
        - re_measures
        - benchmark
        - mixed
        required: true
      code_release:
        type: str
  defences:
    fields:
      id:
        type: str
        required: true
      name:
        type: str
        required: true
      origin:
        type: enum
        values:
        - this_report
        - other_report
        required: true
      base_checkpoint:
        type: str
        required: true
        from_paper: true
      axis1:
        type: enum_list
        values:
        - chat_roles
        - delimiters_markers
        - encodings
        - learned_token_features
        - position
        - internal_intervention
        - no_signal
        required: true
      axis2:
        type: enum_list
        values:
        - prompting
        - soft_tokens_frozen
        - sft
        - preference_optimisation
        - distillation
        - adversarial_rl
        - fixed_inference_rule
        required: true
      components:
        type: list
      isolation:
        type: enum
        values:
        - ablation_present
        - no_ablation
        - not_applicable
        required: true
      layout:
        type: str
        from_paper: true
      signal_dependence:
        type: enum
        values:
        - measured
        - stated
        - not_reported
        - not_applicable
        required: true
      layout_compat:
        type: map
        keys:
        - user_turn
        - separate_role
        - system_context
        - agent_tool_result
        values:
        - supported
        - unsupported
        - untested
      released_weights:
        type: str
      trains_instruction_pair:
        type: enum
        values:
        - 'yes'
        - partly
        - 'no'
        - not_applicable
        required: true
      trains_carry_through:
        type: enum
        values:
        - yes_exact_targets
        - yes_by_preference
        - 'no'
        - not_applicable
        required: true
      page:
        type: page
      anchor:
        type: str
  evaluations:
    fields:
      id:
        type: str
        required: true
      defence_id:
        type: str
        required: true
      benchmark:
        type: str
        required: true
        from_paper: true
      split:
        type: str
      attacker_type:
        type: enum
        values:
        - static
        - adaptive
        - optimisation
        - human
        - rl
        - none
        required: true
      attacker_access:
        type: enum
        values:
        - black_box
        - grey_box
        - white_box
        - not_applicable
        - unreported
        required: true
      attacker_budget:
        type: str
      judge:
        type: str
        required: true
      denominator:
        type: str
      decoding:
        type: str
      sample_size:
        type: str
      layout_same_as_defence:
        type: enum
        values:
        - 'yes'
        - 'no'
        - unclear
        required: true
      measured_by:
        type: str
        required: true
      page:
        type: page
  results:
    fields:
      id:
        type: str
        required: true
      evaluation_id:
        type: str
        required: true
      quantity:
        type: enum
        values:
        - R1
        - R2clean
        - R2attack
        - R3ns
        - R3cp
        - R3safe
        - general_utility
        - other
        required: true
      coverage:
        type: enum
        values:
        - direct
        - proxy
        required: true
      metric_as_printed:
        type: str
        required: true
        from_paper: true
      value:
        type: str
        required: true
        from_paper: true
      unit:
        type: enum
        values:
        - percent
        - fraction
        - score
        - count
        required: true
      direction:
        type: enum
        values:
        - higher_better
        - lower_better
        required: true
      location:
        type: str
      page:
        type: page
  comparisons:
    fields:
      id:
        type: str
        required: true
      left:
        type: str
        required: true
      right:
        type: str
        required: true
      purpose:
        type: enum
        values:
        - W
        - M
        - X
        required: true
      c1_base:
        type: enum
        values:
        - matched
        - unmatched
        - unreported
        - manipulated
        required: true
      c2_layout:
        type: enum
        values:
        - matched
        - unmatched
        - unreported
        - manipulated
        required: true
      c3_benchmark:
        type: enum
        values:
        - matched
        - unmatched
        - unreported
        - manipulated
        required: true
      c4_attacker:
        type: enum
        values:
        - matched
        - unmatched
        - unreported
        - manipulated
        required: true
      c5_judge:
        type: enum
        values:
        - matched
        - unmatched
        - unreported
        - manipulated
        required: true
      c6_decoding:
        type: enum
        values:
        - matched
        - unmatched
        - unreported
        - manipulated
        required: true
      page:
        type: page
  claims:
    fields:
      id:
        type: str
        required: true
      text:
        type: str
        required: true
      location:
        type: str
        required: true
      anchor:
        type: str
        required: true
        from_paper: true
      jobs:
        type: enum_list
        values:
        - R1
        - R2
        - R3
        - general_utility
        required: true
      supported_by:
        type: list
      page:
        type: page
  coverage:
    one: true
    fields:
      R1:
        type: enum
        values:
        - direct
        - proxy
        - not_reported
        - unclear
        - not_applicable
        required: true
      R2clean:
        type: enum
        values:
        - direct
        - proxy
        - not_reported
        - unclear
        - not_applicable
        required: true
      R2attack:
        type: enum
        values:
        - direct
        - proxy
        - not_reported
        - unclear
        - not_applicable
        required: true
      R3ns:
        type: enum
        values:
        - direct
        - proxy
        - not_reported
        - unclear
        - not_applicable
        required: true
      R3cp:
        type: enum
        values:
        - direct
        - proxy
        - not_reported
        - unclear
        - not_applicable
        required: true
      R3safe:
        type: enum
        values:
        - direct
        - proxy
        - not_reported
        - unclear
        - not_applicable
        required: true
  schema_gaps:
    fields:
      where:
        type: str
        required: true
      note:
        type: str
        required: true

In [ ]:
%%writefile schema/coding_rules.md
# Coding rules for the record files (frozen with schema v1 in notebook 03)

These rules say how a paper is turned into a record file in `reports/records/<report_id>.yaml`. They were committed before full extraction (notebook 04). The schema in `schema/record_schema.yaml` says which fields exist and which values they take; these rules say how to choose the values.

## 1. Reading and pages

1. Every record is filled from the PDF in `data/defences/`, whose SHA-256 is in `reports/defences.csv`. Nothing is filled from memory, from an abstract alone or from another paper's description of the work.
2. Every record holding a fact from the paper carries `page`, the 1-based page index of the PDF (not the printed page number). A fact on a different page goes in `pages: {field: page}`.
3. `extraction_scope` says what was coded: "whole report", or the sections and table rows coded. A quantity outside the scope is never coded "not reported".
4. `read: F` means the full text was read. Main-table rows need F.

## 2. Defences

5. One defence record per method or variant on one base checkpoint. A method from another paper that this report runs is a defence record with `origin: other_report`.
6. `name` is the method's own name. When the paper prints one name but runs another checkpoint (SecFid prints "SecAlign" and runs facebook/Meta-SecAlign-8B), `name_as_printed` keeps the printed name and `checkpoint_id` names what was run.
7. Axis 1 (how the source is shown at inference) and axis 2 (how the behaviour was built) take every value that applies. A fixed, non-learned change to the data tokens' embeddings (ASIDE) is `fixed_embedding_transform`, not a learned token feature.
8. `isolation: ablation_present` only when the paper runs the defence with a component removed or changed and reports the result.
9. Signal dependence: `measured` (the defence run with its signal removed or moved, with numbers); `measured_other_checkpoint` (measured, but on another checkpoint of the method, named in `signal_dependence_detail`); `stated` (the paper says the signal is needed, without numbers); `baseline_only` (a prompting defence whose signal-removed condition is just the instruction-only baseline); `not_reported`; `not_applicable` (no inference-time signal).
10. Layout compatibility marks only layouts the paper ran (`supported` or `unsupported`); everything else is `untested`.
11. Amendment 1 (training targets): `trains_instruction_pair` is yes when training pairs the user's own instruction with the same injected words, partly when some injected strings also appear as legitimate instructions with other data, no otherwise. `trains_carry_through` is yes_exact_targets or yes_by_preference when training rewards carrying instruction-bearing text through as data. Both are `not_applicable` when nothing is trained: prompting defences and fixed inference rules, including a steering direction fitted from examples.

## 3. Evaluations

12. One evaluation per configuration under one protocol: one defence (or undefended model), one benchmark split, one attacker, one judge. An injection position reported separately (start, middle, end) is a separate evaluation.
13. The undefended reference is `defence_id: none:<base>`, with `baseline_kind`: off_the_shelf; same_data_twin (instruction-tuned by the authors on the same data, as ASIDE's Vanilla); instruction_prompt_only.
14. `measured_by` is the report whose authors ran the evaluation. A re-measurement of a released checkpoint carries `same_checkpoint_as: <report_id>:<defence_id>` only when the text says the released checkpoint was used.
15. Attacker: static (fixed injections), optimisation (gradient or search on this model), adaptive (built against this defence), human, rl. Access and budget as stated; "unreported" when not given.
16. Decoding as stated. When both arms of a within-report comparison ran in one harness and the paper does not print the setting, the condition is `matched_unstated`.
17. `run` names the run when a paper reports more than one run of the same configuration (canonical, comparison battery, replicate).
18. `subset_of` names the parent set when an evaluation uses a subset; `locked_split` says whether the test split was fixed before testing.
19. `judge_reading` is strict or relaxed when the paper defines both; otherwise single, or absent when not stated.

## 4. Results and the outcome coding

20. One result per printed number, with the metric name as printed and the value as printed. `value_source` says where it is printed. A result shown only in a chart is `value_source: figure_only` with no value; it counts for coverage and is never used in a comparison or a derived number.
21. `qualifier`: exact; approximate ("about 50%"); upper_bound ("below 3%"); lower_bound (CounterSteer's compromise rates under a generation budget).
22. A value relative to a reference uses `unit: percent_of_reference` and names the reference in `reference` (SEP utility set to 50.0 for the undefended model; benign utility as a percentage of the same row's clean run).
23. `spread` keeps the printed spread and the number of runs. Printed differences between two results are not coded; they are recomputed from the results.
24. Quantities: R1 from attack success, compromise or execution rates (R1 = 1 − rate). R2clean is the user's task done on clean data; R2attack under attack. R3ns is non-suppression (SecFid's fidelity, 1 − ignored rate); R3cp is correct handling of the data the task asked for (SecFid's processed rate); R3safe is processed and not executed.
25. AlpacaEval, MMLU and similar general benchmarks are `general_utility`, never R2. SEP utility and "utility under attack" scored on the user's task are R2 proxies unless the paper scores the user's instruction directly. AgentDojo's task checker scores the user's task and is R2 direct. A summary-quality score is at most an R3 proxy. The SEP separation score maps to no single quantity and is `other`.
26. When one printed metric is two quantities at once (SecFid's processed rate in translation and editing is R3cp and an R2attack proxy), the first goes in `quantity` and the second in `also_quantity`.
27. A string-matched attack success that also fires on quoted text is E or Rp in the outcome coding; it is coded Rp when the task asked for the text.

## 5. Comparisons and grades

28. A comparison has a purpose: W (which whole configuration does better), M (does this component cause the change) or X (a reading across two reports). `varies` says what differs between the arms.
29. Six conditions: c1 base checkpoint, c2 layout, c3 benchmark and split, c4 attacker, c5 judge and denominator, c6 decoding. Each is matched, matched_unstated, unmatched, unreported or manipulated (the one condition an M comparison varies on purpose).
30. Grade (plan v2 equation 4), computed by code: A when every condition is matched or is the one manipulated condition; B when exactly one is unmatched; C when two or more are; U when any is unreported. In W the layout may differ by design and is left out. matched_unstated counts as matched within one report and as unreported in an X comparison.

## 6. Claims and coverage

31. A claim is a headline claim from the abstract or introduction, with a verbatim anchor of at most 14 words, the jobs it speaks to, the comparisons that support it and, in `contradicted_by`, any result in the same report that cuts against it.
32. Coverage per quantity: direct; proxy (instrument named); aggregate_only (reported only pooled over defences, models or tasks); not_reported (inside the extraction scope); unclear; not_applicable. `where` points to the table or section.
33. A task partition that changes how a result should be read (CounterSteer's delegated-authority tasks) is described in a residual note; the schema has no field for it.

## 7. What is checked by code

34. Allowed values, required fields, references between records, page within the PDF, every printed value found on its page (in the plain text or in a column-preserving extraction, so table cells are not glued together), every anchor found on its page (a page break is allowed), anchors at most 14 words, and the PDF's SHA-256 against the manifest.

In [ ]:
%%writefile schema/pilot_changes.md
# Schema v0 to v1: the one revision after the six-report pilot

The pilot coded six reports with schema v0 (`schema/record_schema_v0.yaml`, records in `reports/pilot_v0/`). Every place where v0 needed a free-text workaround was written to that record's `schema_gaps` (33 in all). Each change below answers one or more of them. The six records were then recoded with v1 (`reports/records/`) and v1 was frozen.

| # | Change in v1 | Gap it answers (record) |
|---|---|---|
| 1 | results.qualifier: exact, approximate, upper_bound, lower_bound | approximate baselines (Spotlighting r1, r3); compromise rates that are lower bounds under a generation budget (CounterSteer) |
| 2 | results.value_source, with figure_only allowed without a value | clean-task results shown only as charts (Spotlighting Figures 7 and 8) |
| 3 | defences.signal_dependence gains baseline_only and measured_other_checkpoint; signal_dependence_detail | prompting defences whose signal-removed condition is the instruction-only baseline (Spotlighting); rotation ablation run on another checkpoint (ASIDE, Qwen3-8B) |
| 4 | defences.pages and evaluations.pages: a page for each field read elsewhere | facts on several pages in one record (SecAlign pp. 7, 8, 11) |
| 5 | condition value matched_unstated; grade rule counts it as matched within a report and unreported across reports | decoding not printed although both arms ran in one harness (SecAlign, Jia et al., ASIDE's SEP runs) |
| 6 | comparisons.varies (required) | a comparison that varies the training method rather than a condition (SecAlign against StruQ) |
| 7 | results.unit percent_of_reference and results.reference | SEP utility set to 50.0 for the undefended model (SecAlign); utility as a percentage of the row's own clean run (CounterSteer); a win rate against the undefended model (Jia et al.) |
| 8 | evaluations.injection_position, with one evaluation per position | SEP attack success reported at the start, middle and end of the data (SecAlign) |
| 9 | evaluations.baseline_kind | the undefended reference is an instruction-tuned twin on the same data (ASIDE's Vanilla), not an off-the-shelf model |
| 10 | results.spread | means over three runs with standard deviations or standard errors (ASIDE) |
| 11 | evaluations.run | a canonical run and a comparison-battery run of the same configuration (CounterSteer) |
| 12 | defences.name_as_printed and defences.checkpoint_id | "SecAlign" printed while facebook/Meta-SecAlign-8B is run (SecFid) |
| 13 | evaluations.same_checkpoint_as | a re-measurement of a released checkpoint that another report coded (Jia et al. on SecAlign's release) |
| 14 | results.also_quantity | one printed metric that is two quantities (SecFid's processed rate: R3cp and an R2attack proxy) |
| 15 | coverage value aggregate_only, and coverage.where | safe processing reported only by task and framing (SecFid); utility under attack in an uncoded appendix table (CounterSteer) |
| 16 | report.extraction_scope (required) and report.later_version | only part of a report coded in the pilot (Jia et al. Section 5.1); a later peer-reviewed version not read (Spotlighting) |
| 17 | claims.contradicted_by | a no-loss claim that the same report's SEP Utility cuts against (ASIDE) |
| 18 | evaluations.subset_of and evaluations.locked_split | GCG runs on a 50-tuple subset (Jia et al.); a locked test split and a held-out task (SecFid) |
| 19 | evaluations.judge_reading | strict and relaxed success criteria defined in the same paper (Spotlighting, SecAlign Table 8) |
| 20 | axis-1 value fixed_embedding_transform | ASIDE's fixed rotation is not a learned token feature |
| 21 | rule, not field: amendment-1 fields are not_applicable when nothing is trained | a fitted steering direction (CounterSteer) |
| 22 | rule, not field: printed differences are recomputed, not coded; the SEP separation score is quantity other | Table 1 differences in parentheses (SecFid); SEP score (ASIDE, SecAlign) |

Kept as residual notes, not fields (2): CounterSteer's split of surviving compromises into delegated-authority tasks and the rest (a reading aid specific to one paper); SecAlign's SEP utility, which is not stated to be scored on clean or attacked inputs (a reading problem, not a schema gap).

Also fixed during the pilot, in the checker rather than the schema: numbers in table rows are looked for in a column-preserving extraction as well as the plain text, because the plain text glues adjacent cells together (".475.07994.4"); YAML files are read with a loader that keeps yes and no as words (standard YAML 1.1 turns them into true and false); anchors are limited to 14 words.

The six pilot records under the frozen schema.

In [ ]:
%%writefile reports/records/spotlighting_arxiv2024.yaml
report:
  id: spotlighting_arxiv2024
  slug: '2403.14720'
  title: Defending Against Indirect Prompt Injection Attacks With Spotlighting
  first_author: Keegan Hines
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2403.14720'
  version_read: arXiv v1
  first_date: '2024-03-20'
  read: F
  role: proposes
  code_release: none found in the paper
  extraction_scope: whole report
  later_version: CAMLIS 2024, CEUR Workshop Proceedings vol. 3920 (cited by later papers; not read)
defences:
- id: spot_delimit_gpt35
  name: Spotlighting by delimiting
  origin: this_report
  base_checkpoint: GPT-3.5-Turbo (June 2023 version)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  components:
  - system-prompt instruction not to obey text between the symbols
  - begin and end symbols around the document
  isolation: ablation_present
  layout: instruction and document both in the system prompt; document between << and >>
  signal_dependence: measured
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  released_weights: not applicable (API model)
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  anchor: Including specialized delimiters to mark the beginning and end of the input document
  checkpoint_id: GPT-3.5-Turbo (June 2023 version) via the OpenAI API
  signal_dependence_detail: instructions alone against instructions with delimiters (Figure 3), values shown in the figure only
- id: spot_datamark_gpt35
  name: Spotlighting by datamarking
  origin: this_report
  base_checkpoint: GPT-3.5-Turbo (June 2023 version)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  components:
  - system-prompt instruction
  - a marking character between every word of the document
  isolation: no_ablation
  layout: instruction and document both in the system prompt; whitespace in the document replaced by a marking character
  signal_dependence: baseline_only
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  released_weights: not applicable (API model)
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  anchor: we interleave a special token throughout the entirety of the text
  checkpoint_id: GPT-3.5-Turbo (June 2023 version) via the OpenAI API
  signal_dependence_detail: the signal-removed condition is the instruction-only system prompt (Figure 2), not a separate run
- id: spot_datamark_text003
  name: Spotlighting by datamarking
  origin: this_report
  base_checkpoint: text-davinci-003
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: no_ablation
  layout: as spot_datamark_gpt35
  signal_dependence: baseline_only
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  released_weights: not applicable (API model)
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  checkpoint_id: text-davinci-003 via the OpenAI API
  signal_dependence_detail: the signal-removed condition is the instruction-only system prompt (Figure 2), not a separate run
- id: spot_datamark_gpt4
  name: Spotlighting by datamarking
  origin: this_report
  base_checkpoint: GPT-4 (June 2023 version)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: no_ablation
  layout: as spot_datamark_gpt35
  signal_dependence: baseline_only
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  released_weights: not applicable (API model)
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  checkpoint_id: GPT-4 (June 2023 version) via the OpenAI API
  signal_dependence_detail: the signal-removed condition is the instruction-only system prompt (Figure 2), not a separate run
- id: spot_encode_gpt35
  name: Spotlighting by encoding (base64)
  origin: this_report
  base_checkpoint: GPT-3.5-Turbo (June 2023 version)
  axis1:
  - encodings
  axis2:
  - prompting
  components:
  - system-prompt instruction
  - the document encoded in base64
  isolation: no_ablation
  layout: instruction and encoded document both in the system prompt
  signal_dependence: baseline_only
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  released_weights: not applicable (API model)
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  anchor: the input text is transformed using a well-known encoding algorithm such as base64
  checkpoint_id: GPT-3.5-Turbo (June 2023 version) via the OpenAI API
  signal_dependence_detail: the signal-removed condition is the instruction-only system prompt (Figure 2), not a separate run
evaluations:
- id: ev_gpt35_sum_none
  defence_id: none:GPT-3.5-Turbo
  benchmark: authors' synthetic corpus of 1000 documents with keyword-payload injections, summarisation task
  attacker_type: static
  attacker_access: black_box
  attacker_budget: one fixed payload per document
  judge: payload keyword returned in place of the task (Appendix 8.1)
  denominator: attack documents in the corpus
  decoding: temperature 1.0
  sample_size: '1000'
  layout_same_as_defence: 'no'
  measured_by: spotlighting_arxiv2024
  page: 3
  baseline_kind: off_the_shelf
- id: ev_gpt35_sum_datamark
  defence_id: spot_datamark_gpt35
  benchmark: authors' synthetic corpus of 1000 documents with keyword-payload injections, summarisation task
  attacker_type: static
  attacker_access: black_box
  attacker_budget: one fixed payload per document
  judge: payload keyword returned in place of the task (Appendix 8.1)
  denominator: attack documents in the corpus
  decoding: temperature 1.0
  sample_size: '1000'
  layout_same_as_defence: 'yes'
  measured_by: spotlighting_arxiv2024
  page: 4
- id: ev_text003_sum_none
  defence_id: none:text-davinci-003
  benchmark: authors' synthetic corpus, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: payload keyword returned in place of the task
  decoding: temperature 1.0
  sample_size: '1000'
  layout_same_as_defence: 'no'
  measured_by: spotlighting_arxiv2024
  page: 4
  baseline_kind: off_the_shelf
- id: ev_text003_sum_datamark
  defence_id: spot_datamark_text003
  benchmark: authors' synthetic corpus, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: payload keyword returned in place of the task
  decoding: temperature 1.0
  sample_size: '1000'
  layout_same_as_defence: 'yes'
  measured_by: spotlighting_arxiv2024
  page: 4
- id: ev_gpt35_qa_datamark
  defence_id: spot_datamark_gpt35
  benchmark: authors' synthetic corpus, document Q&A task
  attacker_type: static
  attacker_access: black_box
  judge: payload keyword returned in place of the task
  decoding: temperature 1.0
  layout_same_as_defence: 'yes'
  measured_by: spotlighting_arxiv2024
  page: 4
- id: ev_gpt4_qa_datamark
  defence_id: spot_datamark_gpt4
  benchmark: authors' synthetic corpus, document Q&A task
  attacker_type: static
  attacker_access: black_box
  judge: payload keyword returned in place of the task
  decoding: temperature 1.0
  layout_same_as_defence: 'yes'
  measured_by: spotlighting_arxiv2024
  page: 4
- id: ev_gpt35_sum_encode
  defence_id: spot_encode_gpt35
  benchmark: authors' synthetic corpus, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: payload keyword returned in place of the task
  decoding: temperature 1.0
  layout_same_as_defence: 'yes'
  measured_by: spotlighting_arxiv2024
  page: 5
- id: ev_gpt35_qa_encode
  defence_id: spot_encode_gpt35
  benchmark: authors' synthetic corpus, document Q&A task
  attacker_type: static
  attacker_access: black_box
  judge: payload keyword returned in place of the task
  decoding: temperature 1.0
  layout_same_as_defence: 'yes'
  measured_by: spotlighting_arxiv2024
  page: 5
- id: ev_gpt35_nlp_datamark
  defence_id: spot_datamark_gpt35
  benchmark: SQuAD, IMDB sentiment, SuperGLUE WiC and BoolQ with datamarked inputs, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scoring
  layout_same_as_defence: 'yes'
  measured_by: spotlighting_arxiv2024
  page: 5
results:
- id: r1
  evaluation_id: ev_gpt35_sum_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 50%
  unit: percent
  direction: lower_better
  location: Section 5.1 text
  page: 4
  qualifier: approximate
  value_source: text
- id: r2
  evaluation_id: ev_gpt35_sum_datamark
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 3.10%
  unit: percent
  direction: lower_better
  location: Figure 4 caption
  page: 4
  qualifier: exact
  value_source: figure_caption
- id: r3
  evaluation_id: ev_text003_sum_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 40%
  unit: percent
  direction: lower_better
  location: Section 5.1 text
  page: 4
  qualifier: approximate
  value_source: text
- id: r4
  evaluation_id: ev_text003_sum_datamark
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 0.00%
  unit: percent
  direction: lower_better
  location: Figure 4 caption
  page: 4
  qualifier: exact
  value_source: figure_caption
- id: r5
  evaluation_id: ev_gpt35_qa_datamark
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 8.0%
  unit: percent
  direction: lower_better
  location: Figure 5 caption
  page: 4
  qualifier: exact
  value_source: figure_caption
- id: r6
  evaluation_id: ev_gpt4_qa_datamark
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 1.0%
  unit: percent
  direction: lower_better
  location: Figure 5 caption
  page: 4
  qualifier: exact
  value_source: figure_caption
- id: r7
  evaluation_id: ev_gpt35_sum_encode
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 0.0%
  unit: percent
  direction: lower_better
  location: Figure 6 caption
  page: 5
  qualifier: exact
  value_source: figure_caption
- id: r8
  evaluation_id: ev_gpt35_qa_encode
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 1.8%
  unit: percent
  direction: lower_better
  location: Figure 6 caption
  page: 5
  qualifier: exact
  value_source: figure_caption
- id: r9
  evaluation_id: ev_gpt35_nlp_datamark
  quantity: R2clean
  coverage: proxy
  metric_as_printed: task performance on SQuAD, IMDB, WiC and BoolQ with and without datamarking
  unit: percent
  direction: higher_better
  location: Figure 7
  qualifier: approximate
  value_source: figure_only
  page: 5
comparisons:
- id: c1
  left: ev_gpt35_sum_datamark
  right: ev_gpt35_sum_none
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 4
  varies: the datamarking transformation and its system-prompt instruction (the layout)
- id: c2
  left: ev_text003_sum_datamark
  right: ev_text003_sum_none
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 4
  varies: the datamarking transformation and its system-prompt instruction (the layout)
claims:
- id: k1
  text: spotlighting cuts attack success from over 50% to under 2% with GPT-family models
  location: abstract
  anchor: reduces the attack success rate from greater than 50% to below
  jobs:
  - R1
  supported_by:
  - c1
  - c2
  page: 1
- id: k2
  text: minimal detrimental impact on underlying NLP tasks
  location: abstract
  anchor: minimal detrimental impact to underlying
  jobs:
  - R2
  supported_by: []
  page: 1
coverage:
  R1: direct
  R2clean: proxy
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Figures 3 to 6
    R2clean: Figures 7 and 8 (no printed values)

In [ ]:
%%writefile reports/records/secalign_ccs2025.yaml
report:
  id: secalign_ccs2025
  slug: '2410.05451'
  title: 'SecAlign: Defending Against Prompt Injection with Preference Optimization'
  first_author: Sizhe Chen
  venue: ACM CCS 2025
  peer_reviewed: 'yes'
  doi: 10.1145/3719027.3744836
  arxiv: '2410.05451'
  version_read: arXiv v3 (carries the CCS 2025 proceedings header)
  first_date: '2024-10-07'
  read: F
  role: proposes
  code_release: linked from the abstract
  extraction_scope: 'Sections 4.1 to 4.4: Instruct-model rows of Table 6 and Table 3 (base-model rows and Tables 2, 4, 5, 7, 8 not coded in the pilot)'
defences:
- id: secalign_mistral7bi
  name: SecAlign
  origin: this_report
  base_checkpoint: Mistral-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  components:
  - the model's own prompt template
  - DPO on prompt-injected inputs with the legitimate response preferred (LoRA)
  isolation: ablation_present
  layout: instruction and data inside one [INST] block, data separated from the instruction by a single space
  signal_dependence: stated
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: code linked; weights not checked
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: We apply SecAlign to five open-weight models
  signal_dependence_detail: the models have to be used with the exact prompt format (p. 8); no numbers
  pages:
    signal_dependence: 8
    isolation: 11
- id: secalign_llama3_8bi
  name: SecAlign
  origin: this_report
  base_checkpoint: Llama3-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  components:
  - instruction in the system role, data in the user role
  - DPO on prompt-injected inputs with the legitimate response preferred (LoRA)
  isolation: ablation_present
  layout: instruction in the system role, data in the user role
  signal_dependence: stated
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: supported
  released_weights: code linked; weights not checked
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: We apply SecAlign to five open-weight models
  signal_dependence_detail: the models have to be used with the exact prompt format (p. 8); no numbers
  pages:
    signal_dependence: 8
    isolation: 11
- id: struq_mistral7bi
  name: StruQ (reproduced from its released code)
  origin: other_report
  base_checkpoint: Mistral-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  layout: as secalign_mistral7bi
  signal_dependence: not_reported
  released_weights: not stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  anchor: We reproduce StruQ [3] exactly using the released code
- id: struq_llama3_8bi
  name: StruQ (reproduced from its released code)
  origin: other_report
  base_checkpoint: Llama3-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - sft
  isolation: not_applicable
  layout: as secalign_llama3_8bi
  signal_dependence: not_reported
  released_weights: not stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  anchor: We reproduce StruQ [3] exactly using the released code
evaluations:
- id: ev_m_none_free
  defence_id: none:Mistral-7B-Instruct
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: maximum over Ignore, Completion and Ignore-Completion
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  sample_size: '208'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
  baseline_kind: off_the_shelf
  judge_reading: strict
  injection_position: end
- id: ev_m_struq_free
  defence_id: struq_mistral7bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: maximum over Ignore, Completion and Ignore-Completion
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  sample_size: '208'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
  judge_reading: strict
  injection_position: end
- id: ev_m_sec_free
  defence_id: secalign_mistral7bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: maximum over Ignore, Completion and Ignore-Completion
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  sample_size: '208'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
  judge_reading: strict
  injection_position: end
- id: ev_m_none_opt
  defence_id: none:Mistral-7B-Instruct
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: maximum over GCG, AdvPrompter and NeuralExec; GCG plotted to 500 steps (Figure 5)
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
  baseline_kind: off_the_shelf
  judge_reading: strict
  injection_position: end
- id: ev_m_struq_opt
  defence_id: struq_mistral7bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: maximum over GCG, AdvPrompter and NeuralExec
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
  judge_reading: strict
  injection_position: end
- id: ev_m_sec_opt
  defence_id: secalign_mistral7bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: maximum over GCG, AdvPrompter and NeuralExec
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
  judge_reading: strict
  injection_position: end
- id: ev_l_none_free
  defence_id: none:Llama3-8B-Instruct
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: maximum over Ignore, Completion and Ignore-Completion
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  sample_size: '208'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
  baseline_kind: off_the_shelf
  judge_reading: strict
  injection_position: end
- id: ev_l_struq_free
  defence_id: struq_llama3_8bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: maximum over Ignore, Completion and Ignore-Completion
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  sample_size: '208'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
  judge_reading: strict
  injection_position: end
- id: ev_l_sec_free
  defence_id: secalign_llama3_8bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: maximum over Ignore, Completion and Ignore-Completion
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  sample_size: '208'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
  judge_reading: strict
  injection_position: end
- id: ev_l_none_opt
  defence_id: none:Llama3-8B-Instruct
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: maximum over GCG, AdvPrompter and NeuralExec
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
  baseline_kind: off_the_shelf
  judge_reading: strict
  injection_position: end
- id: ev_l_struq_opt
  defence_id: struq_llama3_8bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: maximum over GCG, AdvPrompter and NeuralExec
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
  judge_reading: strict
  injection_position: end
- id: ev_l_sec_opt
  defence_id: secalign_llama3_8bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: maximum over GCG, AdvPrompter and NeuralExec
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
  judge_reading: strict
  injection_position: end
- id: ev_m_none_util
  defence_id: none:Mistral-7B-Instruct
  benchmark: AlpacaEval2 on all 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4-turbo against davinci003 reference replies
  decoding: not reported
  sample_size: '805'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
  baseline_kind: off_the_shelf
- id: ev_m_sec_util
  defence_id: secalign_mistral7bi
  benchmark: AlpacaEval2 on all 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4-turbo against davinci003 reference replies
  decoding: not reported
  sample_size: '805'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_l_none_util
  defence_id: none:Llama3-8B-Instruct
  benchmark: AlpacaEval2 on all 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4-turbo against davinci003 reference replies
  decoding: not reported
  sample_size: '805'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
  baseline_kind: off_the_shelf
- id: ev_l_sec_util
  defence_id: secalign_llama3_8bi
  benchmark: AlpacaEval2 on all 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4-turbo against davinci003 reference replies
  decoding: not reported
  sample_size: '805'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_l_none_injecagent
  defence_id: none:Llama3-8B-Instruct
  benchmark: InjecAgent
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent's own criterion
  layout_same_as_defence: unclear
  measured_by: secalign_ccs2025
  page: 10
  baseline_kind: off_the_shelf
- id: ev_l_sec_injecagent
  defence_id: secalign_llama3_8bi
  benchmark: InjecAgent
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent's own criterion
  layout_same_as_defence: unclear
  measured_by: secalign_ccs2025
  page: 10
- benchmark: SEP (9.1K samples, one unique injection task each)
  attacker_type: static
  attacker_access: black_box
  judge: SEP's own criterion
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 10
  id: ev_l_none_sep
  defence_id: none:Llama3-8B-Instruct
  attacker_budget: Ignore attack
  injection_position: varied
  baseline_kind: off_the_shelf
- benchmark: SEP (9.1K samples, one unique injection task each)
  attacker_type: static
  attacker_access: black_box
  judge: SEP's own criterion
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 10
  id: ev_l_sec_sep
  defence_id: secalign_llama3_8bi
  attacker_budget: Ignore attack
  injection_position: varied
- benchmark: SEP (9.1K samples, one unique injection task each)
  attacker_type: static
  attacker_access: black_box
  judge: SEP's own criterion
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 10
  id: ev_l_none_sep_end
  defence_id: none:Llama3-8B-Instruct
  attacker_budget: Ignore attack with the injection at the end of the data
  injection_position: end
  baseline_kind: off_the_shelf
- benchmark: SEP (9.1K samples, one unique injection task each)
  attacker_type: static
  attacker_access: black_box
  judge: SEP's own criterion
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 10
  id: ev_l_sec_sep_end
  defence_id: secalign_llama3_8bi
  attacker_budget: Ignore attack with the injection at the end of the data
  injection_position: end
- benchmark: SEP (9.1K samples, one unique injection task each)
  attacker_type: static
  attacker_access: black_box
  judge: SEP's own criterion
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 10
  id: ev_l_sec_sep_start
  defence_id: secalign_llama3_8bi
  attacker_budget: Ignore attack with the injection at the start of the data
  injection_position: start
results:
- id: r_m_none_free
  evaluation_id: ev_m_none_free
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Free
  value: '59'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_m_struq_free
  evaluation_id: ev_m_struq_free
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Free
  value: '2'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_m_sec_free
  evaluation_id: ev_m_sec_free
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Free
  value: '0'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_m_none_opt
  evaluation_id: ev_m_none_opt
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Based
  value: '89'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_m_struq_opt
  evaluation_id: ev_m_struq_opt
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Based
  value: '27'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_m_sec_opt
  evaluation_id: ev_m_sec_opt
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Based
  value: '1'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_l_none_free
  evaluation_id: ev_l_none_free
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Free
  value: '51'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_l_struq_free
  evaluation_id: ev_l_struq_free
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Free
  value: '0'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_l_sec_free
  evaluation_id: ev_l_sec_free
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Free
  value: '0'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_l_none_opt
  evaluation_id: ev_l_none_opt
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Based
  value: '97'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_l_struq_opt
  evaluation_id: ev_l_struq_opt
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Based
  value: '45'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_l_sec_opt
  evaluation_id: ev_l_sec_opt
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Based
  value: '8'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_m_none_util
  evaluation_id: ev_m_none_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate
  value: '67.01'
  unit: percent
  direction: higher_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_m_sec_util
  evaluation_id: ev_m_sec_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate
  value: '69.22'
  unit: percent
  direction: higher_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_l_none_util
  evaluation_id: ev_l_none_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate
  value: '85.39'
  unit: percent
  direction: higher_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_l_sec_util
  evaluation_id: ev_l_sec_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate
  value: '85.88'
  unit: percent
  direction: higher_better
  location: Table 6
  page: 15
  qualifier: exact
  value_source: table
- id: r_l_none_sep_util
  evaluation_id: ev_l_none_sep
  quantity: other
  coverage: proxy
  metric_as_printed: SEP Utility (w.r.t. None)
  value: '50.0'
  unit: percent_of_reference
  direction: higher_better
  location: Table 3
  page: 10
  qualifier: exact
  value_source: table
  reference: the undefended Llama3-8B-Instruct, set to 50.0
- id: r_l_sec_sep_util
  evaluation_id: ev_l_sec_sep
  quantity: other
  coverage: proxy
  metric_as_printed: SEP Utility (w.r.t. None)
  value: '46.6'
  unit: percent_of_reference
  direction: higher_better
  location: Table 3
  page: 10
  qualifier: exact
  value_source: table
  reference: the undefended Llama3-8B-Instruct, set to 50.0
- id: r_l_none_sep_end
  evaluation_id: ev_l_none_sep_end
  quantity: R1
  coverage: direct
  metric_as_printed: SEP Ignore ASR (end)
  value: '64.0'
  unit: percent
  direction: lower_better
  location: Table 3
  page: 10
  qualifier: exact
  value_source: table
- id: r_l_sec_sep_end
  evaluation_id: ev_l_sec_sep_end
  quantity: R1
  coverage: direct
  metric_as_printed: SEP Ignore ASR (end)
  value: '6.5'
  unit: percent
  direction: lower_better
  location: Table 3
  page: 10
  qualifier: exact
  value_source: table
- id: r_l_sec_sep_start
  evaluation_id: ev_l_sec_sep_start
  quantity: R1
  coverage: direct
  metric_as_printed: SEP Ignore ASR (start)
  value: '11.0'
  unit: percent
  direction: lower_better
  location: Table 3
  page: 10
  qualifier: exact
  value_source: table
- id: r_l_none_ia
  evaluation_id: ev_l_none_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '75.9'
  unit: percent
  direction: lower_better
  location: Table 3
  page: 10
  qualifier: exact
  value_source: table
- id: r_l_sec_ia
  evaluation_id: ev_l_sec_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '0'
  unit: percent
  direction: lower_better
  location: Table 3
  page: 10
  qualifier: exact
  value_source: table
comparisons:
- id: c_m_free
  left: ev_m_sec_free
  right: ev_m_none_free
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  page: 15
  varies: the defence (defended against undefended arm)
- id: c_m_opt
  left: ev_m_sec_opt
  right: ev_m_none_opt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  page: 15
  varies: the defence (defended against undefended arm)
- id: c_l_opt
  left: ev_l_sec_opt
  right: ev_l_none_opt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  page: 15
  varies: the defence (defended against undefended arm)
- id: c_l_struq_opt
  left: ev_l_sec_opt
  right: ev_l_struq_opt
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  page: 15
  varies: 'training method: DPO (SecAlign) against supervised fine-tuning (StruQ), same base and template'
- id: c_m_util
  left: ev_m_sec_util
  right: ev_m_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  page: 15
  varies: the defence (defended against undefended arm)
- id: c_l_util
  left: ev_l_sec_util
  right: ev_l_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  page: 15
  varies: the defence (defended against undefended arm)
claims:
- id: k1
  text: first method that brings the success of various prompt injections below 10%, including attacks unseen in training
  location: abstract
  anchor: reduces the success rates of various prompt injections to <10%
  jobs:
  - R1
  supported_by:
  - c_m_free
  - c_m_opt
  - c_l_opt
  page: 1
- id: k2
  text: SecAlign models keep similar utility to the model before defensive training
  location: abstract
  anchor: similar utility to the one before defensive training
  jobs:
  - general_utility
  supported_by:
  - c_m_util
  - c_l_util
  page: 1
coverage:
  R1: direct
  R2clean: unclear
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 6, Table 3
    R2clean: Table 3 (SEP utility, clean or attacked not stated)
schema_gaps:
- where: results r_l_*_sep_util
  note: SEP utility is not stated to be scored on clean or attacked inputs, so it stays quantity other and R2clean coverage unclear (a reading problem, not a schema gap)

In [ ]:
%%writefile reports/records/aside_iclr2026.yaml
report:
  id: aside_iclr2026
  slug: '2503.10566'
  title: 'ASIDE: Architectural Separation of Instructions and Data in Language Models'
  first_author: Egor Zverev
  venue: ICLR 2026
  peer_reviewed: 'yes'
  arxiv: '2503.10566'
  version_read: arXiv v4 (marked as published at ICLR 2026)
  first_date: '2025-03-13'
  read: F
  role: proposes
  code_release: not checked in this pass
  extraction_scope: Sections 4 and 5.1; Llama 3.1 8B and Mistral 7B v0.3 rows of Tables 1 and 3 (other models, direct-injection columns and Section 6 not coded in the pilot)
defences:
- id: aside_llama31_8b
  name: ASIDE
  origin: this_report
  base_checkpoint: Llama 3.1 8B (pretrained), instruction-tuned by the authors
  axis1:
  - fixed_embedding_transform
  axis2:
  - sft
  components:
  - fixed orthogonal rotation of the embeddings of data tokens
  - standard instruction tuning without injected examples
  isolation: ablation_present
  layout: 'Alpaca template: instruction with the template''s system prompt in the instruction part; the input and any injection go through the rotated data embedding'
  signal_dependence: measured_other_checkpoint
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  released_weights: not checked in this pass
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  anchor: we apply a fixed (i.e., not learnable) orthogonal rotation
  signal_dependence_detail: rotation-angle ablation on Qwen3-8B only (Figure 7, values in the figure)
  pages:
    signal_dependence: 17
- id: aside_mistral7b
  name: ASIDE
  origin: this_report
  base_checkpoint: Mistral 7B v0.3 (pretrained), instruction-tuned by the authors
  axis1:
  - fixed_embedding_transform
  axis2:
  - sft
  isolation: ablation_present
  layout: as aside_llama31_8b
  signal_dependence: measured_other_checkpoint
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  released_weights: not checked in this pass
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  signal_dependence_detail: rotation-angle ablation on Qwen3-8B only (Figure 7, values in the figure)
  pages:
    signal_dependence: 17
- id: ise_llama31_8b
  name: ISE (re-implemented by the ASIDE authors)
  origin: other_report
  base_checkpoint: Llama 3.1 8B (pretrained), instruction-tuned by the authors
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  layout: as aside_llama31_8b
  signal_dependence: not_reported
  released_weights: not checked in this pass
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
evaluations:
- id: ev_l_van_bipia_text
  defence_id: none:Llama 3.1 8B, instruction-tuned with the same data (Vanilla)
  benchmark: BIPIA text split (programmatically scored scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: BIPIA's programmatic success rule
  decoding: greedy, up to 1024 tokens
  sample_size: 3 independent runs
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
  baseline_kind: same_data_twin
- id: ev_l_aside_bipia_text
  defence_id: aside_llama31_8b
  benchmark: BIPIA text split (programmatically scored scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: BIPIA's programmatic success rule
  decoding: greedy, up to 1024 tokens
  sample_size: 3 independent runs
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
- id: ev_l_ise_bipia_text
  defence_id: ise_llama31_8b
  benchmark: BIPIA text split (programmatically scored scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: BIPIA's programmatic success rule
  decoding: greedy, up to 1024 tokens
  sample_size: 3 independent runs
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
- id: ev_l_van_struq_ood
  defence_id: none:Llama 3.1 8B, instruction-tuned with the same data (Vanilla)
  benchmark: Structured Queries benchmark, out-of-domain (following StruQ)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: Naive, Ignore, Escape and Completion attacks, averaged
  judge: StruQ's success rule (Print exactly Hacked)
  decoding: greedy, up to 1024 tokens
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
  baseline_kind: same_data_twin
- id: ev_l_aside_struq_ood
  defence_id: aside_llama31_8b
  benchmark: Structured Queries benchmark, out-of-domain (following StruQ)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: Naive, Ignore, Escape and Completion attacks, averaged
  judge: StruQ's success rule (Print exactly Hacked)
  decoding: greedy, up to 1024 tokens
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
- id: ev_m_van_bipia_text
  defence_id: none:Mistral 7B v0.3, instruction-tuned with the same data (Vanilla)
  benchmark: BIPIA text split (programmatically scored scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: BIPIA's programmatic success rule
  decoding: greedy, up to 1024 tokens
  sample_size: 3 independent runs
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
  baseline_kind: same_data_twin
- id: ev_m_aside_bipia_text
  defence_id: aside_mistral7b
  benchmark: BIPIA text split (programmatically scored scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: BIPIA's programmatic success rule
  decoding: greedy, up to 1024 tokens
  sample_size: 3 independent runs
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
- id: ev_m_van_struq_id
  defence_id: none:Mistral 7B v0.3, instruction-tuned with the same data (Vanilla)
  benchmark: Structured Queries benchmark, in-domain (following StruQ)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: Naive, Ignore, Escape and Completion attacks, averaged
  judge: StruQ's success rule (Print exactly Hacked)
  decoding: greedy, up to 1024 tokens
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
  baseline_kind: same_data_twin
- id: ev_m_aside_struq_id
  defence_id: aside_mistral7b
  benchmark: Structured Queries benchmark, in-domain (following StruQ)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: Naive, Ignore, Escape and Completion attacks, averaged
  judge: StruQ's success rule (Print exactly Hacked)
  decoding: greedy, up to 1024 tokens
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
- id: ev_l_van_sep
  defence_id: none:Llama 3.1 8B, instruction-tuned with the same data (Vanilla)
  benchmark: SEP (9160 instruction and input pairs) and AlpacaEval 1.0
  attacker_type: static
  attacker_access: black_box
  attacker_budget: one unrelated probe instruction per pair
  judge: SEP probe execution rule; AlpacaEval GPT-4 judge against text-davinci-003
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 23
  baseline_kind: same_data_twin
- id: ev_l_aside_sep
  defence_id: aside_llama31_8b
  benchmark: SEP (9160 instruction and input pairs) and AlpacaEval 1.0
  attacker_type: static
  attacker_access: black_box
  attacker_budget: one unrelated probe instruction per pair
  judge: SEP probe execution rule; AlpacaEval GPT-4 judge against text-davinci-003
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 23
- id: ev_m_van_sep
  defence_id: none:Mistral 7B v0.3, instruction-tuned with the same data (Vanilla)
  benchmark: SEP (9160 instruction and input pairs) and AlpacaEval 1.0
  attacker_type: static
  attacker_access: black_box
  judge: SEP probe execution rule; AlpacaEval GPT-4 judge
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 23
  baseline_kind: same_data_twin
- id: ev_m_aside_sep
  defence_id: aside_mistral7b
  benchmark: SEP (9160 instruction and input pairs) and AlpacaEval 1.0
  attacker_type: static
  attacker_access: black_box
  judge: SEP probe execution rule; AlpacaEval GPT-4 judge
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 23
results:
- id: r1
  evaluation_id: ev_l_van_bipia_text
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (BIPIA-text)
  value: '13.6'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
  qualifier: exact
  value_source: table
  spread: mean and standard deviation over 3 independent runs
- id: r2
  evaluation_id: ev_l_aside_bipia_text
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (BIPIA-text)
  value: '4.1'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
  qualifier: exact
  value_source: table
  spread: mean and standard deviation over 3 independent runs
- id: r3
  evaluation_id: ev_l_ise_bipia_text
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (BIPIA-text)
  value: '11.0'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
  qualifier: exact
  value_source: table
  spread: mean and standard deviation over 3 independent runs
- id: r4
  evaluation_id: ev_l_van_struq_ood
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (StruQ-OOD)
  value: '50.5'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
  qualifier: exact
  value_source: table
  spread: mean and standard deviation over 3 independent runs
- id: r5
  evaluation_id: ev_l_aside_struq_ood
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (StruQ-OOD)
  value: '47.3'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
  qualifier: exact
  value_source: table
  spread: mean and standard deviation over 3 independent runs
- id: r6
  evaluation_id: ev_m_van_bipia_text
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (BIPIA-text)
  value: '11.1'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
  qualifier: exact
  value_source: table
  spread: mean and standard deviation over 3 independent runs
- id: r7
  evaluation_id: ev_m_aside_bipia_text
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (BIPIA-text)
  value: '0.5'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
  qualifier: exact
  value_source: table
  spread: mean and standard deviation over 3 independent runs
- id: r8
  evaluation_id: ev_m_van_struq_id
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (StruQ-ID)
  value: '33.4'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
  qualifier: exact
  value_source: table
  spread: mean and standard deviation over 3 independent runs
- id: r9
  evaluation_id: ev_m_aside_struq_id
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (StruQ-ID)
  value: '9.6'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
  qualifier: exact
  value_source: table
  spread: mean and standard deviation over 3 independent runs
- id: r10
  evaluation_id: ev_l_van_sep
  quantity: other
  coverage: proxy
  metric_as_printed: SEP [%]
  value: '53.2'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
  qualifier: exact
  value_source: table
  spread: standard error of the mean
- id: r11
  evaluation_id: ev_l_aside_sep
  quantity: other
  coverage: proxy
  metric_as_printed: SEP [%]
  value: '83.1'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
  qualifier: exact
  value_source: table
  spread: standard error of the mean
- id: r12
  evaluation_id: ev_l_van_sep
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP Utility [%]
  value: '70.3'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
  qualifier: exact
  value_source: table
  spread: standard error of the mean
- id: r13
  evaluation_id: ev_l_aside_sep
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP Utility [%]
  value: '63.0'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
  qualifier: exact
  value_source: table
  spread: standard error of the mean
- id: r14
  evaluation_id: ev_l_van_sep
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval [%]
  value: '89.3'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
  qualifier: exact
  value_source: table
  spread: standard error of the mean
- id: r15
  evaluation_id: ev_l_aside_sep
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval [%]
  value: '86.6'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
  qualifier: exact
  value_source: table
  spread: standard error of the mean
- id: r16
  evaluation_id: ev_m_van_sep
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP Utility [%]
  value: '45.0'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
  qualifier: exact
  value_source: table
  spread: standard error of the mean
- id: r17
  evaluation_id: ev_m_aside_sep
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP Utility [%]
  value: '25.3'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
  qualifier: exact
  value_source: table
  spread: standard error of the mean
- id: r18
  evaluation_id: ev_m_van_sep
  quantity: other
  coverage: proxy
  metric_as_printed: SEP [%]
  value: '48.0'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
  qualifier: exact
  value_source: table
  spread: standard error of the mean
- id: r19
  evaluation_id: ev_m_aside_sep
  quantity: other
  coverage: proxy
  metric_as_printed: SEP [%]
  value: '92.1'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
  qualifier: exact
  value_source: table
  spread: standard error of the mean
- id: r20
  evaluation_id: ev_m_van_sep
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval [%]
  value: '44.3'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
  qualifier: exact
  value_source: table
  spread: standard error of the mean
- id: r21
  evaluation_id: ev_m_aside_sep
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval [%]
  value: '45.4'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
  qualifier: exact
  value_source: table
  spread: standard error of the mean
comparisons:
- id: c_l_bipia
  left: ev_l_aside_bipia_text
  right: ev_l_van_bipia_text
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 7
  varies: the fixed rotation of data-token embeddings (ASIDE) against the same-data instruction-tuned twin (Vanilla)
- id: c_m_bipia
  left: ev_m_aside_bipia_text
  right: ev_m_van_bipia_text
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 7
  varies: the fixed rotation of data-token embeddings (ASIDE) against the same-data instruction-tuned twin (Vanilla)
- id: c_m_struq
  left: ev_m_aside_struq_id
  right: ev_m_van_struq_id
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 7
  varies: the fixed rotation of data-token embeddings (ASIDE) against the same-data instruction-tuned twin (Vanilla)
- id: c_l_sep
  left: ev_l_aside_sep
  right: ev_l_van_sep
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  page: 23
  varies: the fixed rotation of data-token embeddings (ASIDE) against the same-data instruction-tuned twin (Vanilla)
- id: c_m_sep
  left: ev_m_aside_sep
  right: ev_m_van_sep
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  page: 23
  varies: the fixed rotation of data-token embeddings (ASIDE) against the same-data instruction-tuned twin (Vanilla)
claims:
- id: k1
  text: instruction-tuning with ASIDE gives much higher instruction-data separation without performance loss
  location: abstract
  anchor: achieves substantially higher instruction-data separation without performance loss
  jobs:
  - general_utility
  - R2
  supported_by:
  - c_l_sep
  - c_m_sep
  page: 1
  contradicted_by:
  - r13
  - r17
- id: k2
  text: ASIDE makes models more robust on prompt injection benchmarks without dedicated safety training
  location: abstract
  anchor: makes the models more robust to prompt injection benchmarks, even without dedicated safety training
  jobs:
  - R1
  supported_by:
  - c_l_bipia
  - c_m_bipia
  - c_m_struq
  page: 1
coverage:
  R1: direct
  R2clean: proxy
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1
    R2clean: Table 3 (SEP Utility)

In [ ]:
%%writefile reports/records/countersteer_arxiv2026.yaml
report:
  id: countersteer_arxiv2026
  slug: '2609.36570'
  title: 'CounterSteer: Suppressing Indirect Prompt Injection with Activation Steering'
  first_author: Mark Russinovich
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2609.36570'
  version_read: arXiv v1
  first_date: '2026-09-29'
  read: F
  role: mixed
  code_release: manifest of the AgentDojo grid stated as released; code not checked in this pass
  extraction_scope: Table II and Table III rows for gpt-oss-20b, Qwen3-30B and Llama-3.1-8B, the abstract claims (appendix tables not coded in the pilot)
defences:
- id: cs_gptoss20b
  name: CounterSteer
  origin: this_report
  base_checkpoint: gpt-oss-20b
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  components:
  - per-model residual-stream direction fitted from paired episodes that differ only in whether an embedded instruction is followed
  - the direction subtracted at every tool-result token during prefill
  isolation: ablation_present
  layout: agent transcript; the edit applies to the tool-result spans, which the serving stack must mark
  signal_dependence: stated
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  released_weights: not applicable (weights unchanged)
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 1
  anchor: only white-box serving and tool-result span boundaries
  signal_dependence_detail: requires tool-result span boundaries from the serving stack; no run with missing or misplaced spans in the pages read
- id: cs_qwen3_30b
  name: CounterSteer
  origin: this_report
  base_checkpoint: Qwen3-30B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  layout: as cs_gptoss20b
  signal_dependence: stated
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  released_weights: not applicable (weights unchanged)
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  signal_dependence_detail: requires tool-result span boundaries from the serving stack; no run with missing or misplaced spans in the pages read
- id: cs_llama31_8b
  name: CounterSteer
  origin: this_report
  base_checkpoint: Llama-3.1-8B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  layout: as cs_gptoss20b
  signal_dependence: stated
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  released_weights: not applicable (weights unchanged)
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  signal_dependence_detail: requires tool-result span boundaries from the serving stack; no run with missing or misplaced spans in the pages read
- id: spot_gptoss20b
  name: Spotlighting (re-measured by CounterSteer)
  origin: other_report
  base_checkpoint: gpt-oss-20b
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  layout: AgentDojo's spotlighting configuration
  signal_dependence: not_reported
  released_weights: not applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 8
evaluations:
- id: ev_g_none_ad
  defence_id: none:gpt-oss-20b
  benchmark: AgentDojo, fixed 180-episode grid over all four suites
  split: 180 of 629 user-by-injection pairs, pinned
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  sample_size: '180'
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 6
  baseline_kind: off_the_shelf
  run: canonical
- id: ev_g_cs_ad
  defence_id: cs_gptoss20b
  benchmark: AgentDojo, fixed 180-episode grid over all four suites
  split: 180 of 629 user-by-injection pairs, pinned
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  sample_size: '180'
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 6
  run: canonical
- id: ev_q_none_ad
  defence_id: none:Qwen3-30B
  benchmark: AgentDojo, fixed 180-episode grid over all four suites
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  sample_size: '180'
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 6
  baseline_kind: off_the_shelf
  run: canonical
- id: ev_q_cs_ad
  defence_id: cs_qwen3_30b
  benchmark: AgentDojo, fixed 180-episode grid over all four suites
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  sample_size: '180'
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 6
  run: canonical
- id: ev_l_none_ad
  defence_id: none:Llama-3.1-8B
  benchmark: AgentDojo, fixed 180-episode grid over all four suites
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  sample_size: '180'
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 6
  baseline_kind: off_the_shelf
  run: canonical
- id: ev_l_cs_ad
  defence_id: cs_llama31_8b
  benchmark: AgentDojo, fixed 180-episode grid over all four suites
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  sample_size: '180'
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 6
  run: canonical
- id: ev_g_none_batt
  defence_id: none:gpt-oss-20b
  benchmark: AgentDojo, fixed 180-episode grid (comparison battery run)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 8
  baseline_kind: off_the_shelf
  run: comparison battery
- id: ev_g_spot_batt
  defence_id: spot_gptoss20b
  benchmark: AgentDojo, fixed 180-episode grid (comparison battery run)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 8
  run: comparison battery
- id: ev_g_cs_batt
  defence_id: cs_gptoss20b
  benchmark: AgentDojo, fixed 180-episode grid (comparison battery run)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 8
  run: comparison battery
- id: ev_g_none_autodojo
  defence_id: none:gpt-oss-20b
  benchmark: AutoDojo over AgentDojo, 36 cells
  attacker_type: adaptive
  attacker_access: black_box
  attacker_budget: six optimiser iterations (AD@6)
  judge: AgentDojo's end-to-end checker
  decoding: greedy, 4096-token per-turn budget
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 8
  baseline_kind: off_the_shelf
  run: canonical
- id: ev_g_cs_autodojo
  defence_id: cs_gptoss20b
  benchmark: AutoDojo over AgentDojo, 36 cells
  attacker_type: adaptive
  attacker_access: black_box
  attacker_budget: six optimiser iterations (AD@6)
  judge: AgentDojo's end-to-end checker
  decoding: greedy, 4096-token per-turn budget
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 8
  run: canonical
results:
- id: r1
  evaluation_id: ev_g_none_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo compromise rate (undefended)
  value: '.475'
  unit: fraction
  direction: lower_better
  location: Table II
  page: 6
  qualifier: lower_bound
  value_source: table
- id: r2
  evaluation_id: ev_g_cs_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo compromise rate (defended)
  value: '.079'
  unit: fraction
  direction: lower_better
  location: Table II
  page: 6
  qualifier: lower_bound
  value_source: table
- id: r3
  evaluation_id: ev_g_cs_ad
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility, % of clean (typography-normalized)
  value: '94.4'
  unit: percent_of_reference
  direction: higher_better
  location: Table II
  page: 6
  qualifier: exact
  value_source: table
  reference: the same row's own clean run (typography-normalized for gpt-oss-20b)
- id: r4
  evaluation_id: ev_q_none_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo compromise rate (undefended)
  value: '.489'
  unit: fraction
  direction: lower_better
  location: Table II
  page: 6
  qualifier: lower_bound
  value_source: table
- id: r5
  evaluation_id: ev_q_cs_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo compromise rate (defended)
  value: '.072'
  unit: fraction
  direction: lower_better
  location: Table II
  page: 6
  qualifier: lower_bound
  value_source: table
- id: r6
  evaluation_id: ev_q_cs_ad
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility, % of clean
  value: '94.1'
  unit: percent_of_reference
  direction: higher_better
  location: Table II
  page: 6
  qualifier: exact
  value_source: table
  reference: the same row's own clean run (typography-normalized for gpt-oss-20b)
- id: r7
  evaluation_id: ev_l_none_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo compromise rate (undefended)
  value: '.100'
  unit: fraction
  direction: lower_better
  location: Table II
  page: 6
  qualifier: lower_bound
  value_source: table
- id: r8
  evaluation_id: ev_l_cs_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo compromise rate (defended)
  value: '.050'
  unit: fraction
  direction: lower_better
  location: Table II
  page: 6
  qualifier: lower_bound
  value_source: table
- id: r9
  evaluation_id: ev_l_cs_ad
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility, % of clean
  value: '100.0'
  unit: percent_of_reference
  direction: higher_better
  location: Table II
  page: 6
  qualifier: exact
  value_source: table
  reference: the same row's own clean run (typography-normalized for gpt-oss-20b)
- id: r10
  evaluation_id: ev_g_none_batt
  quantity: R1
  coverage: direct
  metric_as_printed: cmp (no defense)
  value: '.483'
  unit: fraction
  direction: lower_better
  location: Table III
  page: 8
  qualifier: lower_bound
  value_source: table
- id: r11
  evaluation_id: ev_g_none_batt
  quantity: R2clean
  coverage: direct
  metric_as_printed: 'util (no defense: absolute clean completion)'
  value: '98.1'
  unit: percent
  direction: higher_better
  location: Table III
  page: 8
  qualifier: exact
  value_source: table
- id: r12
  evaluation_id: ev_g_spot_batt
  quantity: R1
  coverage: direct
  metric_as_printed: cmp (spotlighting)
  value: '.420'
  unit: fraction
  direction: lower_better
  location: Table III
  page: 8
  qualifier: lower_bound
  value_source: table
- id: r13
  evaluation_id: ev_g_spot_batt
  quantity: R2clean
  coverage: direct
  metric_as_printed: util (% of clean)
  value: '92.5'
  unit: percent_of_reference
  direction: higher_better
  location: Table III
  page: 8
  qualifier: exact
  value_source: table
  reference: the same row's own clean run (typography-normalized for gpt-oss-20b)
- id: r14
  evaluation_id: ev_g_cs_batt
  quantity: R1
  coverage: direct
  metric_as_printed: cmp (CounterSteer)
  value: '.091'
  unit: fraction
  direction: lower_better
  location: Table III
  page: 8
  qualifier: lower_bound
  value_source: table
- id: r15
  evaluation_id: ev_g_cs_batt
  quantity: R2clean
  coverage: direct
  metric_as_printed: util (% of clean)
  value: '90.6'
  unit: percent_of_reference
  direction: higher_better
  location: Table III
  page: 8
  qualifier: exact
  value_source: table
  reference: the same row's own clean run (typography-normalized for gpt-oss-20b)
- id: r16
  evaluation_id: ev_g_none_autodojo
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo AD@6 (no defense)
  value: '.673'
  unit: fraction
  direction: lower_better
  location: Table III, bottom block
  page: 8
  qualifier: lower_bound
  value_source: table
- id: r17
  evaluation_id: ev_g_cs_autodojo
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo AD@6 (CounterSteer)
  value: '.175'
  unit: fraction
  direction: lower_better
  location: Table III, bottom block
  page: 8
  qualifier: lower_bound
  value_source: table
comparisons:
- id: c_g_ad
  left: ev_g_cs_ad
  right: ev_g_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 6
  varies: the defence (defended against undefended arm)
- id: c_q_ad
  left: ev_q_cs_ad
  right: ev_q_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 6
  varies: the defence (defended against undefended arm)
- id: c_l_ad
  left: ev_l_cs_ad
  right: ev_l_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 6
  varies: the defence (defended against undefended arm)
- id: c_g_spot
  left: ev_g_cs_batt
  right: ev_g_spot_batt
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 8
  varies: 'the defence: CounterSteer against spotlighting on the same base and grid'
- id: c_g_adaptive
  left: ev_g_cs_autodojo
  right: ev_g_none_autodojo
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 8
  varies: the defence under the adaptive attacker
claims:
- id: k1
  text: AgentDojo compromise falls from 0.10-0.49 to 0.006-0.079 at 93-100% normalized benign utility across five models
  location: abstract
  anchor: AgentDojo compromise rate from 0.10–0.49 to 0.006–0.079
  jobs:
  - R1
  - R2
  supported_by:
  - c_g_ad
  - c_q_ad
  - c_l_ad
  page: 1
- id: k2
  text: a benchmark-level adaptive attacker is held to about a quarter of its undefended success on the two most deeply evaluated models
  location: abstract
  anchor: held to roughly a quarter of that on the two most deeply
  jobs:
  - R1
  supported_by:
  - c_g_adaptive
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables II and III
    R2clean: Tables II and III
    R2attack: Table XIV (not coded in the pilot)
schema_gaps:
- where: report
  note: surviving compromises split into delegated-authority tasks and the rest (p. 7); kept as text here and handled by a coding rule, not a field

In [ ]:
%%writefile reports/records/jia_critical_arxiv2025.yaml
report:
  id: jia_critical_arxiv2025
  slug: '2505.18333'
  title: A Critical Evaluation of Defenses against Prompt Injection Attacks
  first_author: Yuqi Jia
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2505.18333'
  version_read: arXiv v1
  first_date: '2025-05-23'
  read: F
  role: re_measures
  code_release: https://github.com/PIEval123/PIEval
  extraction_scope: 'Section 5.1: StruQ and SecAlign on Llama-3-8B-Instruct (Instruction Hierarchy, detectors and Appendix A not coded in the pilot)'
defences:
- id: secalign_llama3_8bi_rel
  name: SecAlign (checkpoint released by its authors)
  origin: other_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  layout: the released model's own delimiters; special tokens filtered from the data at run time
  signal_dependence: not_reported
  released_weights: downloaded from the SecAlign release
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  anchor: We downloaded LLMs released by StruQ and SecAlign
- id: struq_llama3_8bi_rel
  name: StruQ (checkpoint released by its authors)
  origin: other_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  layout: the released model's own delimiters; special tokens filtered from the data at run time
  signal_dependence: not_reported
  released_weights: downloaded from the StruQ release
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  anchor: We downloaded LLMs released by StruQ and SecAlign
evaluations:
- id: ev_none_opi_comb
  defence_id: none:Llama-3-8B-Instruct
  benchmark: OpenPromptInjection (seven NLP tasks)
  split: 4,900 target and injected prompt tuples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: Combined Attack
  judge: reply equivalent to the injected task's reference response (ASV)
  denominator: 4,900 tuples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 9
  baseline_kind: off_the_shelf
- id: ev_sec_opi_comb
  defence_id: secalign_llama3_8bi_rel
  benchmark: OpenPromptInjection (seven NLP tasks)
  split: 4,900 target and injected prompt tuples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: Combined Attack
  judge: reply equivalent to the injected task's reference response (ASV)
  denominator: 4,900 tuples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 9
  same_checkpoint_as: secalign_ccs2025:secalign_llama3_8bi
- id: ev_none_opi_gcg
  defence_id: none:Llama-3-8B-Instruct
  benchmark: OpenPromptInjection (seven NLP tasks)
  split: 50 tuples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: GCG optimising the separator
  judge: reply equivalent to the injected task's reference response (ASV)
  denominator: 50 tuples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 9
  baseline_kind: off_the_shelf
  subset_of: the 4,900 OpenPromptInjection tuples
- id: ev_sec_opi_gcg
  defence_id: secalign_llama3_8bi_rel
  benchmark: OpenPromptInjection (seven NLP tasks)
  split: 50 tuples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: GCG optimising the separator
  judge: reply equivalent to the injected task's reference response (ASV)
  denominator: 50 tuples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 9
  same_checkpoint_as: secalign_ccs2025:secalign_llama3_8bi
  subset_of: the 4,900 OpenPromptInjection tuples
- id: ev_sec_opi_gcg_adapt
  defence_id: secalign_llama3_8bi_rel
  benchmark: OpenPromptInjection (seven NLP tasks)
  split: 50 tuples
  attacker_type: adaptive
  attacker_access: white_box
  attacker_budget: GCG with delimiters built from the nearest tokens to the defence's special tokens
  judge: reply equivalent to the injected task's reference response (ASV)
  denominator: 50 tuples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 9
  same_checkpoint_as: secalign_ccs2025:secalign_llama3_8bi
  subset_of: the 4,900 OpenPromptInjection tuples
- id: ev_struq_opi_gcg
  defence_id: struq_llama3_8bi_rel
  benchmark: OpenPromptInjection (seven NLP tasks)
  split: 50 tuples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: GCG optimising the separator
  judge: reply equivalent to the injected task's reference response (ASV)
  denominator: 50 tuples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 9
  subset_of: the 4,900 OpenPromptInjection tuples
- id: ev_sec_alpaca_rel
  defence_id: secalign_llama3_8bi_rel
  benchmark: AlpacaEval on AlpacaFarm, reference model Llama-3-8B-Instruct
  attacker_type: none
  attacker_access: not_applicable
  judge: AlpacaEval win rate against the undefended Llama-3-8B-Instruct
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 8
  same_checkpoint_as: secalign_ccs2025:secalign_llama3_8bi
- id: ev_none_opi_util
  defence_id: none:Llama-3-8B-Instruct
  benchmark: OpenPromptInjection clean prompt-response pairs
  split: 700 pairs, 100 per task
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy on the target task
  decoding: not reported
  sample_size: '700'
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 8
  baseline_kind: off_the_shelf
- id: ev_sec_opi_util
  defence_id: secalign_llama3_8bi_rel
  benchmark: OpenPromptInjection clean prompt-response pairs
  split: 700 pairs, 100 per task
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy on the target task
  decoding: not reported
  sample_size: '700'
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 8
  same_checkpoint_as: secalign_ccs2025:secalign_llama3_8bi
- id: ev_none_mmlu_util
  defence_id: none:Llama-3-8B-Instruct
  benchmark: MMLU-PI clean pairs
  split: 200 MMLU test prompts
  attacker_type: none
  attacker_access: not_applicable
  judge: MMLU accuracy
  decoding: not reported
  sample_size: '200'
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 8
  baseline_kind: off_the_shelf
- id: ev_sec_mmlu_util
  defence_id: secalign_llama3_8bi_rel
  benchmark: MMLU-PI clean pairs
  split: 200 MMLU test prompts
  attacker_type: none
  attacker_access: not_applicable
  judge: MMLU accuracy
  decoding: not reported
  sample_size: '200'
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 8
  same_checkpoint_as: secalign_ccs2025:secalign_llama3_8bi
results:
- id: r1
  evaluation_id: ev_none_opi_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASV, Combined Attack (existing)
  value: '0.52'
  unit: fraction
  direction: lower_better
  location: Table 2
  page: 9
  qualifier: exact
  value_source: table
- id: r2
  evaluation_id: ev_sec_opi_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASV, Combined Attack (existing)
  value: '0.06'
  unit: fraction
  direction: lower_better
  location: Table 2
  page: 9
  qualifier: exact
  value_source: table
- id: r3
  evaluation_id: ev_none_opi_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASV, GCG (existing)
  value: '1.00'
  unit: fraction
  direction: lower_better
  location: Table 2
  page: 9
  qualifier: exact
  value_source: table
- id: r4
  evaluation_id: ev_sec_opi_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASV, GCG (existing)
  value: '0.24'
  unit: fraction
  direction: lower_better
  location: Table 2
  page: 9
  qualifier: exact
  value_source: table
- id: r5
  evaluation_id: ev_sec_opi_gcg_adapt
  quantity: R1
  coverage: direct
  metric_as_printed: ASV, GCG (adaptive)
  value: '0.46'
  unit: fraction
  direction: lower_better
  location: Table 2
  page: 9
  qualifier: exact
  value_source: table
- id: r6
  evaluation_id: ev_struq_opi_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASV, GCG (existing)
  value: '0.80'
  unit: fraction
  direction: lower_better
  location: Table 2
  page: 9
  qualifier: exact
  value_source: table
- id: r7
  evaluation_id: ev_sec_alpaca_rel
  quantity: general_utility
  coverage: direct
  metric_as_printed: Win Rate (%) against Llama-3-8B-Instruct
  value: '36.42'
  unit: percent
  direction: higher_better
  location: Table 1(a)
  page: 8
  qualifier: exact
  value_source: table
  reference: the undefended Llama-3-8B-Instruct as the AlpacaEval reference model
- id: r8
  evaluation_id: ev_none_opi_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: absolute utility, OpenPromptInjection
  value: '0.65'
  unit: fraction
  direction: higher_better
  location: Table 1(b)
  page: 8
  qualifier: exact
  value_source: table
- id: r9
  evaluation_id: ev_sec_opi_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: absolute utility, OpenPromptInjection
  value: '0.48'
  unit: fraction
  direction: higher_better
  location: Table 1(b)
  page: 8
  qualifier: exact
  value_source: table
- id: r10
  evaluation_id: ev_none_mmlu_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: absolute utility, MMLU-PI
  value: '0.45'
  unit: fraction
  direction: higher_better
  location: Table 1(b)
  page: 8
  qualifier: exact
  value_source: table
- id: r11
  evaluation_id: ev_sec_mmlu_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: absolute utility, MMLU-PI
  value: '0.34'
  unit: fraction
  direction: higher_better
  location: Table 1(b)
  page: 8
  qualifier: exact
  value_source: table
comparisons:
- id: c_comb
  left: ev_sec_opi_comb
  right: ev_none_opi_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  page: 9
  varies: the defence (defended against undefended arm)
- id: c_gcg
  left: ev_sec_opi_gcg
  right: ev_none_opi_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  page: 9
  varies: the defence (defended against undefended arm)
- id: c_adapt
  left: ev_sec_opi_gcg_adapt
  right: ev_sec_opi_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched_unstated
  page: 9
  varies: 'the attacker: adaptive GCG against existing GCG on the same defended model'
- id: c_util
  left: ev_sec_opi_util
  right: ev_none_opi_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  page: 8
  varies: the defence (defended against undefended arm)
claims:
- id: k1
  text: existing defences are not as successful as previously reported when evaluated with diverse prompts, adaptive attacks and absolute utility
  location: abstract
  anchor: we show that existing defenses are not as successful as previously reported
  jobs:
  - R1
  - R2
  - general_utility
  supported_by:
  - c_gcg
  - c_adapt
  - c_util
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 2
    R2clean: Table 1(b), OpenPromptInjection absolute utility

In [ ]:
%%writefile reports/records/secfid_arxiv2026.yaml
report:
  id: secfid_arxiv2026
  slug: '2606.30783'
  title: 'Security–Fidelity Tradeoffs: The Hidden Cost of Prompt Injection Defense'
  first_author: Mitchell Hermon
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2606.30783'
  version_read: arXiv v1
  first_date: '2026-06-29'
  read: F
  role: mixed
  code_release: not checked in this pass
  extraction_scope: 'Sections 3.4 to 4.3: Llama 3.1 8B rows of Table 1 and Table 2 (other models, Tables 3 and 13 to 28 not coded in the pilot)'
defences:
- id: aside_llama31_8b_sf
  name: ASIDE (re-measured by SecFid)
  origin: other_report
  base_checkpoint: Llama 3.1 8B
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  layout: inference through the official ASIDE GitHub repository
  signal_dependence: not_reported
  released_weights: released checkpoint used
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 24
- id: ise_llama31_8b_sf
  name: ISE (re-measured by SecFid)
  origin: other_report
  base_checkpoint: Llama 3.1 8B
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  layout: inference through the official ASIDE GitHub repository
  signal_dependence: not_reported
  released_weights: released checkpoint used
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 24
- id: metasecalign_8b_sf
  name: Meta-SecAlign-8B (re-measured by SecFid)
  origin: other_report
  base_checkpoint: Llama 3.1 8B with the facebook/Meta-SecAlign-8B adapter
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  layout: the inference procedure specified by the defence (data in its own input role)
  signal_dependence: not_reported
  released_weights: facebook/Meta-SecAlign-8B
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 24
  anchor: facebook/Meta-SecAlign-8B
  name_as_printed: SecAlign
  checkpoint_id: facebook/Meta-SecAlign-8B
- id: deftokens_llama31_8b_sf
  name: DefensiveTokens (re-measured by SecFid)
  origin: other_report
  base_checkpoint: Llama 3.1 8B
  axis1:
  - learned_token_features
  axis2:
  - soft_tokens_frozen
  isolation: not_applicable
  layout: five defensive tokens prepended to the input
  signal_dependence: not_reported
  released_weights: released tokens used
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 24
  anchor: DefensiveTokens is evaluated with five defensive tokens prepended to the input
- id: secfid_dpo_8b
  name: Fidelity-aware DPO (SecFid preference tuning)
  origin: this_report
  base_checkpoint: Meta-SecAlign-8B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  components:
  - Meta-SecAlign-8B starting point
  - DPO with processed preferred to ignored and ignored preferred to executed, on counting, extraction and translation
  isolation: ablation_present
  layout: as metasecalign_8b_sf
  signal_dependence: not_reported
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  released_weights: not checked in this pass
  trains_instruction_pair: 'no'
  trains_carry_through: yes_by_preference
  page: 7
  anchor: We build preferences from the three-way labels with the ordering
evaluations:
- id: ev_base
  defence_id: none:Llama 3.1 8B
  benchmark: 'SecFid (1,168 instances: counting, extraction, translation, editing)'
  attacker_type: static
  attacker_access: black_box
  attacker_budget: one probe per instance, plain or persuasive framing, prefix, inside or suffix
  judge: signature answer for execution; set, count or embedding-similarity checks for processed or ignored
  denominator: 1,168 instances
  decoding: temperature 0
  sample_size: '1168'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 6
  baseline_kind: off_the_shelf
- id: ev_aside
  defence_id: aside_llama31_8b_sf
  benchmark: SecFid (1,168 instances)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: as ev_base
  judge: as ev_base
  denominator: 1,168 instances
  decoding: temperature 0
  sample_size: '1168'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 6
- id: ev_ise
  defence_id: ise_llama31_8b_sf
  benchmark: SecFid (1,168 instances)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: as ev_base
  judge: as ev_base
  denominator: 1,168 instances
  decoding: temperature 0
  sample_size: '1168'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 6
- id: ev_msa
  defence_id: metasecalign_8b_sf
  benchmark: SecFid (1,168 instances)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: as ev_base
  judge: as ev_base
  denominator: 1,168 instances
  decoding: temperature 0
  sample_size: '1168'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 6
- id: ev_dt
  defence_id: deftokens_llama31_8b_sf
  benchmark: SecFid (1,168 instances)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: as ev_base
  judge: as ev_base
  denominator: 1,168 instances
  decoding: temperature 0
  sample_size: '1168'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 6
- id: ev_msa_core
  defence_id: metasecalign_8b_sf
  benchmark: SecFid core held-out split (counting, extraction, translation)
  split: 141 locked cases
  attacker_type: static
  attacker_access: black_box
  judge: as ev_base
  denominator: '141'
  decoding: temperature 0
  sample_size: '141'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 7
  locked_split: 'yes'
- id: ev_dpo_core
  defence_id: secfid_dpo_8b
  benchmark: SecFid core held-out split (counting, extraction, translation)
  split: 141 locked cases
  attacker_type: static
  attacker_access: black_box
  judge: as ev_base
  denominator: '141'
  decoding: temperature 0
  sample_size: '141'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 7
  locked_split: 'yes'
- id: ev_msa_edit
  defence_id: metasecalign_8b_sf
  benchmark: SecFid editing task, held out from training
  split: 273 cases
  attacker_type: static
  attacker_access: black_box
  judge: as ev_base
  denominator: '273'
  decoding: temperature 0
  sample_size: '273'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 7
  locked_split: 'yes'
- id: ev_dpo_edit
  defence_id: secfid_dpo_8b
  benchmark: SecFid editing task, held out from training
  split: 273 cases
  attacker_type: static
  attacker_access: black_box
  judge: as ev_base
  denominator: '273'
  decoding: temperature 0
  sample_size: '273'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 7
  locked_split: 'yes'
results:
- id: r_base_e
  evaluation_id: ev_base
  quantity: R1
  coverage: direct
  metric_as_printed: Exec.
  value: '49.1'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
- id: r_base_i
  evaluation_id: ev_base
  quantity: R3ns
  coverage: direct
  metric_as_printed: Ign.
  value: '7.7'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
- id: r_base_p
  evaluation_id: ev_base
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '50.2'
  unit: percent
  direction: higher_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
  also_quantity:
  - R2attack
- id: r_aside_e
  evaluation_id: ev_aside
  quantity: R1
  coverage: direct
  metric_as_printed: Exec.
  value: '5.7'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
- id: r_aside_i
  evaluation_id: ev_aside
  quantity: R3ns
  coverage: direct
  metric_as_printed: Ign.
  value: '24.7'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
- id: r_aside_p
  evaluation_id: ev_aside
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '39.6'
  unit: percent
  direction: higher_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
  also_quantity:
  - R2attack
- id: r_ise_e
  evaluation_id: ev_ise
  quantity: R1
  coverage: direct
  metric_as_printed: Exec.
  value: '14.0'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
- id: r_ise_i
  evaluation_id: ev_ise
  quantity: R3ns
  coverage: direct
  metric_as_printed: Ign.
  value: '20.3'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
- id: r_ise_p
  evaluation_id: ev_ise
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '53.6'
  unit: percent
  direction: higher_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
  also_quantity:
  - R2attack
- id: r_msa_e
  evaluation_id: ev_msa
  quantity: R1
  coverage: direct
  metric_as_printed: Exec.
  value: '0.7'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
- id: r_msa_i
  evaluation_id: ev_msa
  quantity: R3ns
  coverage: direct
  metric_as_printed: Ign.
  value: '26.1'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
- id: r_msa_p
  evaluation_id: ev_msa
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '67.0'
  unit: percent
  direction: higher_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
  also_quantity:
  - R2attack
- id: r_dt_e
  evaluation_id: ev_dt
  quantity: R1
  coverage: direct
  metric_as_printed: Exec.
  value: '1.9'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
- id: r_dt_i
  evaluation_id: ev_dt
  quantity: R3ns
  coverage: direct
  metric_as_printed: Ign.
  value: '53.7'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
- id: r_dt_p
  evaluation_id: ev_dt
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '37.7'
  unit: percent
  direction: higher_better
  location: Table 1
  page: 6
  qualifier: exact
  value_source: table
  also_quantity:
  - R2attack
- id: r_core_msa_i
  evaluation_id: ev_msa_core
  quantity: R3ns
  coverage: direct
  metric_as_printed: Ign.
  value: '17.7'
  unit: percent
  direction: lower_better
  location: Table 2
  page: 7
  qualifier: exact
  value_source: table
- id: r_core_msa_p
  evaluation_id: ev_msa_core
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '73.8'
  unit: percent
  direction: higher_better
  location: Table 2
  page: 7
  qualifier: exact
  value_source: table
  also_quantity:
  - R2attack
- id: r_core_dpo_i
  evaluation_id: ev_dpo_core
  quantity: R3ns
  coverage: direct
  metric_as_printed: Ign.
  value: '9.2'
  unit: percent
  direction: lower_better
  location: Table 2
  page: 7
  qualifier: exact
  value_source: table
- id: r_core_dpo_p
  evaluation_id: ev_dpo_core
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '83.0'
  unit: percent
  direction: higher_better
  location: Table 2
  page: 7
  qualifier: exact
  value_source: table
  also_quantity:
  - R2attack
- id: r_edit_msa_p
  evaluation_id: ev_msa_edit
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '43.2'
  unit: percent
  direction: higher_better
  location: Table 2
  page: 7
  qualifier: exact
  value_source: table
  also_quantity:
  - R2attack
- id: r_edit_dpo_p
  evaluation_id: ev_dpo_edit
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '80.6'
  unit: percent
  direction: higher_better
  location: Table 2
  page: 7
  qualifier: exact
  value_source: table
  also_quantity:
  - R2attack
- id: r_edit_dpo_e
  evaluation_id: ev_dpo_edit
  quantity: R1
  coverage: direct
  metric_as_printed: Exec.
  value: '0.7'
  unit: percent
  direction: lower_better
  location: Table 2
  page: 7
  qualifier: exact
  value_source: table
comparisons:
- id: c_aside
  left: ev_aside
  right: ev_base
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 6
  varies: the defence (defended against undefended arm)
- id: c_msa
  left: ev_msa
  right: ev_base
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 6
  varies: the defence (defended against undefended arm)
- id: c_dt
  left: ev_dt
  right: ev_base
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 6
  varies: the defence (defended against undefended arm)
- id: c_dpo_core
  left: ev_dpo_core
  right: ev_msa_core
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 7
  varies: an added DPO stage with processed preferred to ignored and ignored to executed, on the same Meta-SecAlign-8B start
- id: c_dpo_edit
  left: ev_dpo_edit
  right: ev_msa_edit
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 7
  varies: an added DPO stage with processed preferred to ignored and ignored to executed, on the same Meta-SecAlign-8B start
claims:
- id: k1
  text: defences resist injected instructions largely by suppressing untrusted text, and no model or defence achieves both security and fidelity
  location: abstract
  anchor: defenses resist injected instructions largely by suppressing untrusted text
  jobs:
  - R1
  - R3
  supported_by:
  - c_aside
  - c_msa
  - c_dt
  page: 1
- id: k2
  text: fidelity-aware preference tuning raises processing while keeping security
  location: Section 4.3
  anchor: S EC F ID preference tuning improves processing while
  jobs:
  - R1
  - R3
  supported_by:
  - c_dpo_core
  - c_dpo_edit
  page: 7
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: proxy
  R3ns: direct
  R3cp: direct
  R3safe: aggregate_only
  where:
    R1: Tables 1 and 2
    R3ns: Tables 1 and 2
    R3cp: Tables 1 and 2
    R3safe: Tables 15 to 23, by task and framing only

The same six reports as first coded with schema v0, with their schema gaps.

In [ ]:
%%writefile reports/pilot_v0/spotlighting_arxiv2024.yaml
report:
  id: spotlighting_arxiv2024
  slug: '2403.14720'
  title: Defending Against Indirect Prompt Injection Attacks With Spotlighting
  first_author: Keegan Hines
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2403.14720'
  version_read: arXiv v1
  first_date: '2024-03-20'
  read: F
  role: proposes
  code_release: none found in the paper
defences:
- id: spot_delimit_gpt35
  name: Spotlighting by delimiting
  origin: this_report
  base_checkpoint: GPT-3.5-Turbo (June 2023 version)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  components:
  - system-prompt instruction not to obey text between the symbols
  - begin and end symbols around the document
  isolation: ablation_present
  layout: instruction and document both in the system prompt; document between << and >>
  signal_dependence: measured
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  released_weights: not applicable (API model)
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  anchor: Including specialized delimiters to mark the beginning and end of the input document
- id: spot_datamark_gpt35
  name: Spotlighting by datamarking
  origin: this_report
  base_checkpoint: GPT-3.5-Turbo (June 2023 version)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  components:
  - system-prompt instruction
  - a marking character between every word of the document
  isolation: no_ablation
  layout: instruction and document both in the system prompt; whitespace in the document replaced by a marking character
  signal_dependence: not_reported
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  released_weights: not applicable (API model)
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  anchor: we interleave a special token throughout the entirety of the text
- id: spot_datamark_text003
  name: Spotlighting by datamarking
  origin: this_report
  base_checkpoint: text-davinci-003
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: no_ablation
  layout: as spot_datamark_gpt35
  signal_dependence: not_reported
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  released_weights: not applicable (API model)
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
- id: spot_datamark_gpt4
  name: Spotlighting by datamarking
  origin: this_report
  base_checkpoint: GPT-4 (June 2023 version)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: no_ablation
  layout: as spot_datamark_gpt35
  signal_dependence: not_reported
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  released_weights: not applicable (API model)
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
- id: spot_encode_gpt35
  name: Spotlighting by encoding (base64)
  origin: this_report
  base_checkpoint: GPT-3.5-Turbo (June 2023 version)
  axis1:
  - encodings
  axis2:
  - prompting
  components:
  - system-prompt instruction
  - the document encoded in base64
  isolation: no_ablation
  layout: instruction and encoded document both in the system prompt
  signal_dependence: not_reported
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  released_weights: not applicable (API model)
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  anchor: the input text is transformed using a well-known encoding algorithm such as base64
evaluations:
- id: ev_gpt35_sum_none
  defence_id: none:GPT-3.5-Turbo
  benchmark: authors' synthetic corpus of 1000 documents with keyword-payload injections, summarisation task
  attacker_type: static
  attacker_access: black_box
  attacker_budget: one fixed payload per document
  judge: payload keyword returned in place of the task (Appendix 8.1)
  denominator: attack documents in the corpus
  decoding: temperature 1.0
  sample_size: '1000'
  layout_same_as_defence: 'no'
  measured_by: spotlighting_arxiv2024
  page: 3
- id: ev_gpt35_sum_datamark
  defence_id: spot_datamark_gpt35
  benchmark: authors' synthetic corpus of 1000 documents with keyword-payload injections, summarisation task
  attacker_type: static
  attacker_access: black_box
  attacker_budget: one fixed payload per document
  judge: payload keyword returned in place of the task (Appendix 8.1)
  denominator: attack documents in the corpus
  decoding: temperature 1.0
  sample_size: '1000'
  layout_same_as_defence: 'yes'
  measured_by: spotlighting_arxiv2024
  page: 4
- id: ev_text003_sum_none
  defence_id: none:text-davinci-003
  benchmark: authors' synthetic corpus, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: payload keyword returned in place of the task
  decoding: temperature 1.0
  sample_size: '1000'
  layout_same_as_defence: 'no'
  measured_by: spotlighting_arxiv2024
  page: 4
- id: ev_text003_sum_datamark
  defence_id: spot_datamark_text003
  benchmark: authors' synthetic corpus, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: payload keyword returned in place of the task
  decoding: temperature 1.0
  sample_size: '1000'
  layout_same_as_defence: 'yes'
  measured_by: spotlighting_arxiv2024
  page: 4
- id: ev_gpt35_qa_datamark
  defence_id: spot_datamark_gpt35
  benchmark: authors' synthetic corpus, document Q&A task
  attacker_type: static
  attacker_access: black_box
  judge: payload keyword returned in place of the task
  decoding: temperature 1.0
  layout_same_as_defence: 'yes'
  measured_by: spotlighting_arxiv2024
  page: 4
- id: ev_gpt4_qa_datamark
  defence_id: spot_datamark_gpt4
  benchmark: authors' synthetic corpus, document Q&A task
  attacker_type: static
  attacker_access: black_box
  judge: payload keyword returned in place of the task
  decoding: temperature 1.0
  layout_same_as_defence: 'yes'
  measured_by: spotlighting_arxiv2024
  page: 4
- id: ev_gpt35_sum_encode
  defence_id: spot_encode_gpt35
  benchmark: authors' synthetic corpus, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: payload keyword returned in place of the task
  decoding: temperature 1.0
  layout_same_as_defence: 'yes'
  measured_by: spotlighting_arxiv2024
  page: 5
- id: ev_gpt35_qa_encode
  defence_id: spot_encode_gpt35
  benchmark: authors' synthetic corpus, document Q&A task
  attacker_type: static
  attacker_access: black_box
  judge: payload keyword returned in place of the task
  decoding: temperature 1.0
  layout_same_as_defence: 'yes'
  measured_by: spotlighting_arxiv2024
  page: 5
- id: ev_gpt35_nlp_datamark
  defence_id: spot_datamark_gpt35
  benchmark: SQuAD, IMDB sentiment, SuperGLUE WiC and BoolQ with datamarked inputs, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scoring
  layout_same_as_defence: 'yes'
  measured_by: spotlighting_arxiv2024
  page: 5
results:
- id: r1
  evaluation_id: ev_gpt35_sum_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 50%
  unit: percent
  direction: lower_better
  location: Section 5.1 text
  page: 4
- id: r2
  evaluation_id: ev_gpt35_sum_datamark
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 3.10%
  unit: percent
  direction: lower_better
  location: Figure 4 caption
  page: 4
- id: r3
  evaluation_id: ev_text003_sum_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 40%
  unit: percent
  direction: lower_better
  location: Section 5.1 text
  page: 4
- id: r4
  evaluation_id: ev_text003_sum_datamark
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 0.00%
  unit: percent
  direction: lower_better
  location: Figure 4 caption
  page: 4
- id: r5
  evaluation_id: ev_gpt35_qa_datamark
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 8.0%
  unit: percent
  direction: lower_better
  location: Figure 5 caption
  page: 4
- id: r6
  evaluation_id: ev_gpt4_qa_datamark
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 1.0%
  unit: percent
  direction: lower_better
  location: Figure 5 caption
  page: 4
- id: r7
  evaluation_id: ev_gpt35_sum_encode
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 0.0%
  unit: percent
  direction: lower_better
  location: Figure 6 caption
  page: 5
- id: r8
  evaluation_id: ev_gpt35_qa_encode
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: 1.8%
  unit: percent
  direction: lower_better
  location: Figure 6 caption
  page: 5
comparisons:
- id: c1
  left: ev_gpt35_sum_datamark
  right: ev_gpt35_sum_none
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 4
- id: c2
  left: ev_text003_sum_datamark
  right: ev_text003_sum_none
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 4
claims:
- id: k1
  text: spotlighting cuts attack success from over 50% to under 2% with GPT-family models
  location: abstract
  anchor: reduces the attack success rate from greater than 50% to below
  jobs:
  - R1
  supported_by:
  - c1
  - c2
  page: 1
- id: k2
  text: minimal detrimental impact on underlying NLP tasks
  location: abstract
  anchor: minimal detrimental impact to underlying
  jobs:
  - R2
  supported_by: []
  page: 1
coverage:
  R1: direct
  R2clean: proxy
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
schema_gaps:
- where: results r1, r3
  note: the baselines are printed as approximate values (approximately 50%, 40%); v0 has no way to mark a value as approximate or as a bound (below 3%)
- where: evaluation ev_gpt35_nlp_datamark
  note: the clean-task results are shown only as bar charts without printed numbers (Figures 7 and 8); v0 cannot record that a result exists but is read only from a figure
- where: defences spot_datamark_*, spot_encode_*
  note: for a prompting defence the signal-removed condition coincides with the instruction-only baseline (Figure 2, Figure 3); v0's signal_dependence has no value for that
- where: report
  note: the work was later published (CAMLIS 2024, per citing papers); v0 has no field linking the version read to a later peer-reviewed version
- where: results
  note: the judge has a strict and a relaxed reading of success (Appendix 8.1); v0 has no field for which reading produced the number

In [ ]:
%%writefile reports/pilot_v0/secalign_ccs2025.yaml
report:
  id: secalign_ccs2025
  slug: '2410.05451'
  title: 'SecAlign: Defending Against Prompt Injection with Preference Optimization'
  first_author: Sizhe Chen
  venue: ACM CCS 2025
  peer_reviewed: 'yes'
  doi: 10.1145/3719027.3744836
  arxiv: '2410.05451'
  version_read: arXiv v3 (carries the CCS 2025 proceedings header)
  first_date: '2024-10-07'
  read: F
  role: proposes
  code_release: linked from the abstract
defences:
- id: secalign_mistral7bi
  name: SecAlign
  origin: this_report
  base_checkpoint: Mistral-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  components:
  - the model's own prompt template
  - DPO on prompt-injected inputs with the legitimate response preferred (LoRA)
  isolation: ablation_present
  layout: instruction and data inside one [INST] block, data separated from the instruction by a single space
  signal_dependence: stated
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: code linked; weights not checked
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: We apply SecAlign to five open-weight models
- id: secalign_llama3_8bi
  name: SecAlign
  origin: this_report
  base_checkpoint: Llama3-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  components:
  - instruction in the system role, data in the user role
  - DPO on prompt-injected inputs with the legitimate response preferred (LoRA)
  isolation: ablation_present
  layout: instruction in the system role, data in the user role
  signal_dependence: stated
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: supported
  released_weights: code linked; weights not checked
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: We apply SecAlign to five open-weight models
- id: struq_mistral7bi
  name: StruQ (reproduced from its released code)
  origin: other_report
  base_checkpoint: Mistral-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  layout: as secalign_mistral7bi
  signal_dependence: not_reported
  released_weights: not stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  anchor: We reproduce StruQ [3] exactly using the released code
- id: struq_llama3_8bi
  name: StruQ (reproduced from its released code)
  origin: other_report
  base_checkpoint: Llama3-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - sft
  isolation: not_applicable
  layout: as secalign_llama3_8bi
  signal_dependence: not_reported
  released_weights: not stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  anchor: We reproduce StruQ [3] exactly using the released code
evaluations:
- id: ev_m_none_free
  defence_id: none:Mistral-7B-Instruct
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: maximum over Ignore, Completion and Ignore-Completion
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  sample_size: '208'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_m_struq_free
  defence_id: struq_mistral7bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: maximum over Ignore, Completion and Ignore-Completion
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  sample_size: '208'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_m_sec_free
  defence_id: secalign_mistral7bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: maximum over Ignore, Completion and Ignore-Completion
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  sample_size: '208'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_m_none_opt
  defence_id: none:Mistral-7B-Instruct
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: maximum over GCG, AdvPrompter and NeuralExec; GCG plotted to 500 steps (Figure 5)
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_m_struq_opt
  defence_id: struq_mistral7bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: maximum over GCG, AdvPrompter and NeuralExec
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_m_sec_opt
  defence_id: secalign_mistral7bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: maximum over GCG, AdvPrompter and NeuralExec
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_l_none_free
  defence_id: none:Llama3-8B-Instruct
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: maximum over Ignore, Completion and Ignore-Completion
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  sample_size: '208'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_l_struq_free
  defence_id: struq_llama3_8bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: maximum over Ignore, Completion and Ignore-Completion
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  sample_size: '208'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_l_sec_free
  defence_id: secalign_llama3_8bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: maximum over Ignore, Completion and Ignore-Completion
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  sample_size: '208'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_l_none_opt
  defence_id: none:Llama3-8B-Instruct
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: maximum over GCG, AdvPrompter and NeuralExec
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_l_struq_opt
  defence_id: struq_llama3_8bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: maximum over GCG, AdvPrompter and NeuralExec
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_l_sec_opt
  defence_id: secalign_llama3_8bi
  benchmark: AlpacaFarm samples with a data part
  split: 208 samples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: maximum over GCG, AdvPrompter and NeuralExec
  judge: reply begins with Hacked or hacked
  denominator: 208 samples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_m_none_util
  defence_id: none:Mistral-7B-Instruct
  benchmark: AlpacaEval2 on all 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4-turbo against davinci003 reference replies
  decoding: not reported
  sample_size: '805'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_m_sec_util
  defence_id: secalign_mistral7bi
  benchmark: AlpacaEval2 on all 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4-turbo against davinci003 reference replies
  decoding: not reported
  sample_size: '805'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_l_none_util
  defence_id: none:Llama3-8B-Instruct
  benchmark: AlpacaEval2 on all 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4-turbo against davinci003 reference replies
  decoding: not reported
  sample_size: '805'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_l_sec_util
  defence_id: secalign_llama3_8bi
  benchmark: AlpacaEval2 on all 805 AlpacaFarm samples
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4-turbo against davinci003 reference replies
  decoding: not reported
  sample_size: '805'
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 15
- id: ev_l_none_sep
  defence_id: none:Llama3-8B-Instruct
  benchmark: SEP (9.1K samples, one unique injection task each)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: Ignore attack with the injection at the start, middle or end of the data
  judge: SEP's own criterion
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 10
- id: ev_l_sec_sep
  defence_id: secalign_llama3_8bi
  benchmark: SEP (9.1K samples, one unique injection task each)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: Ignore attack with the injection at the start, middle or end of the data
  judge: SEP's own criterion
  layout_same_as_defence: 'yes'
  measured_by: secalign_ccs2025
  page: 10
- id: ev_l_none_injecagent
  defence_id: none:Llama3-8B-Instruct
  benchmark: InjecAgent
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent's own criterion
  layout_same_as_defence: unclear
  measured_by: secalign_ccs2025
  page: 10
- id: ev_l_sec_injecagent
  defence_id: secalign_llama3_8bi
  benchmark: InjecAgent
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent's own criterion
  layout_same_as_defence: unclear
  measured_by: secalign_ccs2025
  page: 10
results:
- id: r_m_none_free
  evaluation_id: ev_m_none_free
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Free
  value: '59'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
- id: r_m_struq_free
  evaluation_id: ev_m_struq_free
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Free
  value: '2'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
- id: r_m_sec_free
  evaluation_id: ev_m_sec_free
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Free
  value: '0'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
- id: r_m_none_opt
  evaluation_id: ev_m_none_opt
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Based
  value: '89'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
- id: r_m_struq_opt
  evaluation_id: ev_m_struq_opt
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Based
  value: '27'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
- id: r_m_sec_opt
  evaluation_id: ev_m_sec_opt
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Based
  value: '1'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
- id: r_l_none_free
  evaluation_id: ev_l_none_free
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Free
  value: '51'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
- id: r_l_struq_free
  evaluation_id: ev_l_struq_free
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Free
  value: '0'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
- id: r_l_sec_free
  evaluation_id: ev_l_sec_free
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Free
  value: '0'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
- id: r_l_none_opt
  evaluation_id: ev_l_none_opt
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Based
  value: '97'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
- id: r_l_struq_opt
  evaluation_id: ev_l_struq_opt
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Based
  value: '45'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
- id: r_l_sec_opt
  evaluation_id: ev_l_sec_opt
  quantity: R1
  coverage: direct
  metric_as_printed: Max ASR Opt.-Based
  value: '8'
  unit: percent
  direction: lower_better
  location: Table 6
  page: 15
- id: r_m_none_util
  evaluation_id: ev_m_none_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate
  value: '67.01'
  unit: percent
  direction: higher_better
  location: Table 6
  page: 15
- id: r_m_sec_util
  evaluation_id: ev_m_sec_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate
  value: '69.22'
  unit: percent
  direction: higher_better
  location: Table 6
  page: 15
- id: r_l_none_util
  evaluation_id: ev_l_none_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate
  value: '85.39'
  unit: percent
  direction: higher_better
  location: Table 6
  page: 15
- id: r_l_sec_util
  evaluation_id: ev_l_sec_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate
  value: '85.88'
  unit: percent
  direction: higher_better
  location: Table 6
  page: 15
- id: r_l_none_sep_util
  evaluation_id: ev_l_none_sep
  quantity: other
  coverage: proxy
  metric_as_printed: SEP Utility (w.r.t. None)
  value: '50.0'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 10
- id: r_l_sec_sep_util
  evaluation_id: ev_l_sec_sep
  quantity: other
  coverage: proxy
  metric_as_printed: SEP Utility (w.r.t. None)
  value: '46.6'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 10
- id: r_l_none_sep_end
  evaluation_id: ev_l_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP Ignore ASR (end)
  value: '64.0'
  unit: percent
  direction: lower_better
  location: Table 3
  page: 10
- id: r_l_sec_sep_end
  evaluation_id: ev_l_sec_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP Ignore ASR (end)
  value: '6.5'
  unit: percent
  direction: lower_better
  location: Table 3
  page: 10
- id: r_l_sec_sep_start
  evaluation_id: ev_l_sec_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP Ignore ASR (start)
  value: '11.0'
  unit: percent
  direction: lower_better
  location: Table 3
  page: 10
- id: r_l_none_ia
  evaluation_id: ev_l_none_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '75.9'
  unit: percent
  direction: lower_better
  location: Table 3
  page: 10
- id: r_l_sec_ia
  evaluation_id: ev_l_sec_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '0'
  unit: percent
  direction: lower_better
  location: Table 3
  page: 10
comparisons:
- id: c_m_free
  left: ev_m_sec_free
  right: ev_m_none_free
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 15
- id: c_m_opt
  left: ev_m_sec_opt
  right: ev_m_none_opt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 15
- id: c_l_opt
  left: ev_l_sec_opt
  right: ev_l_none_opt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 15
- id: c_l_struq_opt
  left: ev_l_sec_opt
  right: ev_l_struq_opt
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 15
- id: c_m_util
  left: ev_m_sec_util
  right: ev_m_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 15
- id: c_l_util
  left: ev_l_sec_util
  right: ev_l_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 15
claims:
- id: k1
  text: first method that brings the success of various prompt injections below 10%, including attacks unseen in training
  location: abstract
  anchor: reduces the success rates of various prompt injections to <10%
  jobs:
  - R1
  supported_by:
  - c_m_free
  - c_m_opt
  - c_l_opt
  page: 1
- id: k2
  text: SecAlign models keep similar utility to the model before defensive training
  location: abstract
  anchor: similar utility to the one before defensive training
  jobs:
  - general_utility
  supported_by:
  - c_m_util
  - c_l_util
  page: 1
coverage:
  R1: direct
  R2clean: unclear
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
schema_gaps:
- where: comparisons c_*
  note: decoding settings are not reported anywhere in the paper; both arms ran in one harness, so c6 is coded matched; v0 cannot separate matched and stated from matched by construction with the value unreported
- where: comparison c_l_struq_opt
  note: the varied element is the training method (DPO against SFT), not one of the six conditions; v0 has no field naming what a mechanism comparison varies
- where: defences secalign_*
  note: facts come from several pages (models p7, format dependence p8, ablation p11); v0 allows one page per record
- where: results r_l_*_sep_util
  note: SEP utility is reported relative to the undefended model (50.0 by construction) and it is not stated whether it is scored on clean or attacked inputs; v0 has no relative-to-baseline unit and no clean/attacked flag,
    so it is coded other and R2clean coverage unclear
- where: results r_l_*_sep_*
  note: SEP ASR is reported for three injection positions (start, middle, end); v0 has no field for the position of the injection inside the data
- where: claim k2
  note: 'the utility claim rests on AlpacaEval2 and other general benchmarks only (coding rule: general utility, never R2); v0 records this through jobs but has no explicit flag'

In [ ]:
%%writefile reports/pilot_v0/aside_iclr2026.yaml
report:
  id: aside_iclr2026
  slug: '2503.10566'
  title: 'ASIDE: Architectural Separation of Instructions and Data in Language Models'
  first_author: Egor Zverev
  venue: ICLR 2026
  peer_reviewed: 'yes'
  arxiv: '2503.10566'
  version_read: arXiv v4 (marked as published at ICLR 2026)
  first_date: '2025-03-13'
  read: F
  role: proposes
  code_release: not checked in this pass
defences:
- id: aside_llama31_8b
  name: ASIDE
  origin: this_report
  base_checkpoint: Llama 3.1 8B (pretrained), instruction-tuned by the authors
  axis1:
  - learned_token_features
  axis2:
  - sft
  components:
  - fixed orthogonal rotation of the embeddings of data tokens
  - standard instruction tuning without injected examples
  isolation: ablation_present
  layout: 'Alpaca template: instruction with the template''s system prompt in the instruction part; the input and any injection go through the rotated data embedding'
  signal_dependence: not_reported
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  released_weights: not checked in this pass
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  anchor: we apply a fixed (i.e., not learnable) orthogonal rotation
- id: aside_mistral7b
  name: ASIDE
  origin: this_report
  base_checkpoint: Mistral 7B v0.3 (pretrained), instruction-tuned by the authors
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: ablation_present
  layout: as aside_llama31_8b
  signal_dependence: not_reported
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  released_weights: not checked in this pass
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
- id: ise_llama31_8b
  name: ISE (re-implemented by the ASIDE authors)
  origin: other_report
  base_checkpoint: Llama 3.1 8B (pretrained), instruction-tuned by the authors
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  layout: as aside_llama31_8b
  signal_dependence: not_reported
  released_weights: not checked in this pass
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
evaluations:
- id: ev_l_van_bipia_text
  defence_id: none:Llama 3.1 8B, instruction-tuned with the same data (Vanilla)
  benchmark: BIPIA text split (programmatically scored scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: BIPIA's programmatic success rule
  decoding: greedy, up to 1024 tokens
  sample_size: 3 independent runs
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
- id: ev_l_aside_bipia_text
  defence_id: aside_llama31_8b
  benchmark: BIPIA text split (programmatically scored scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: BIPIA's programmatic success rule
  decoding: greedy, up to 1024 tokens
  sample_size: 3 independent runs
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
- id: ev_l_ise_bipia_text
  defence_id: ise_llama31_8b
  benchmark: BIPIA text split (programmatically scored scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: BIPIA's programmatic success rule
  decoding: greedy, up to 1024 tokens
  sample_size: 3 independent runs
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
- id: ev_l_van_struq_ood
  defence_id: none:Llama 3.1 8B, instruction-tuned with the same data (Vanilla)
  benchmark: Structured Queries benchmark, out-of-domain (following StruQ)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: Naive, Ignore, Escape and Completion attacks, averaged
  judge: StruQ's success rule (Print exactly Hacked)
  decoding: greedy, up to 1024 tokens
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
- id: ev_l_aside_struq_ood
  defence_id: aside_llama31_8b
  benchmark: Structured Queries benchmark, out-of-domain (following StruQ)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: Naive, Ignore, Escape and Completion attacks, averaged
  judge: StruQ's success rule (Print exactly Hacked)
  decoding: greedy, up to 1024 tokens
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
- id: ev_m_van_bipia_text
  defence_id: none:Mistral 7B v0.3, instruction-tuned with the same data (Vanilla)
  benchmark: BIPIA text split (programmatically scored scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: BIPIA's programmatic success rule
  decoding: greedy, up to 1024 tokens
  sample_size: 3 independent runs
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
- id: ev_m_aside_bipia_text
  defence_id: aside_mistral7b
  benchmark: BIPIA text split (programmatically scored scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: BIPIA's programmatic success rule
  decoding: greedy, up to 1024 tokens
  sample_size: 3 independent runs
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
- id: ev_m_van_struq_id
  defence_id: none:Mistral 7B v0.3, instruction-tuned with the same data (Vanilla)
  benchmark: Structured Queries benchmark, in-domain (following StruQ)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: Naive, Ignore, Escape and Completion attacks, averaged
  judge: StruQ's success rule (Print exactly Hacked)
  decoding: greedy, up to 1024 tokens
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
- id: ev_m_aside_struq_id
  defence_id: aside_mistral7b
  benchmark: Structured Queries benchmark, in-domain (following StruQ)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: Naive, Ignore, Escape and Completion attacks, averaged
  judge: StruQ's success rule (Print exactly Hacked)
  decoding: greedy, up to 1024 tokens
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 7
- id: ev_l_van_sep
  defence_id: none:Llama 3.1 8B, instruction-tuned with the same data (Vanilla)
  benchmark: SEP (9160 instruction and input pairs) and AlpacaEval 1.0
  attacker_type: static
  attacker_access: black_box
  attacker_budget: one unrelated probe instruction per pair
  judge: SEP probe execution rule; AlpacaEval GPT-4 judge against text-davinci-003
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 23
- id: ev_l_aside_sep
  defence_id: aside_llama31_8b
  benchmark: SEP (9160 instruction and input pairs) and AlpacaEval 1.0
  attacker_type: static
  attacker_access: black_box
  attacker_budget: one unrelated probe instruction per pair
  judge: SEP probe execution rule; AlpacaEval GPT-4 judge against text-davinci-003
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 23
- id: ev_m_van_sep
  defence_id: none:Mistral 7B v0.3, instruction-tuned with the same data (Vanilla)
  benchmark: SEP (9160 instruction and input pairs) and AlpacaEval 1.0
  attacker_type: static
  attacker_access: black_box
  judge: SEP probe execution rule; AlpacaEval GPT-4 judge
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 23
- id: ev_m_aside_sep
  defence_id: aside_mistral7b
  benchmark: SEP (9160 instruction and input pairs) and AlpacaEval 1.0
  attacker_type: static
  attacker_access: black_box
  judge: SEP probe execution rule; AlpacaEval GPT-4 judge
  layout_same_as_defence: 'yes'
  measured_by: aside_iclr2026
  page: 23
results:
- id: r1
  evaluation_id: ev_l_van_bipia_text
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (BIPIA-text)
  value: '13.6'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
- id: r2
  evaluation_id: ev_l_aside_bipia_text
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (BIPIA-text)
  value: '4.1'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
- id: r3
  evaluation_id: ev_l_ise_bipia_text
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (BIPIA-text)
  value: '11.0'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
- id: r4
  evaluation_id: ev_l_van_struq_ood
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (StruQ-OOD)
  value: '50.5'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
- id: r5
  evaluation_id: ev_l_aside_struq_ood
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (StruQ-OOD)
  value: '47.3'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
- id: r6
  evaluation_id: ev_m_van_bipia_text
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (BIPIA-text)
  value: '11.1'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
- id: r7
  evaluation_id: ev_m_aside_bipia_text
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (BIPIA-text)
  value: '0.5'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
- id: r8
  evaluation_id: ev_m_van_struq_id
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (StruQ-ID)
  value: '33.4'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
- id: r9
  evaluation_id: ev_m_aside_struq_id
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (StruQ-ID)
  value: '9.6'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 7
- id: r10
  evaluation_id: ev_l_van_sep
  quantity: other
  coverage: proxy
  metric_as_printed: SEP [%]
  value: '53.2'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
- id: r11
  evaluation_id: ev_l_aside_sep
  quantity: other
  coverage: proxy
  metric_as_printed: SEP [%]
  value: '83.1'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
- id: r12
  evaluation_id: ev_l_van_sep
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP Utility [%]
  value: '70.3'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
- id: r13
  evaluation_id: ev_l_aside_sep
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP Utility [%]
  value: '63.0'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
- id: r14
  evaluation_id: ev_l_van_sep
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval [%]
  value: '89.3'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
- id: r15
  evaluation_id: ev_l_aside_sep
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval [%]
  value: '86.6'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
- id: r16
  evaluation_id: ev_m_van_sep
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP Utility [%]
  value: '45.0'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
- id: r17
  evaluation_id: ev_m_aside_sep
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP Utility [%]
  value: '25.3'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
- id: r18
  evaluation_id: ev_m_van_sep
  quantity: other
  coverage: proxy
  metric_as_printed: SEP [%]
  value: '48.0'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
- id: r19
  evaluation_id: ev_m_aside_sep
  quantity: other
  coverage: proxy
  metric_as_printed: SEP [%]
  value: '92.1'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
- id: r20
  evaluation_id: ev_m_van_sep
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval [%]
  value: '44.3'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
- id: r21
  evaluation_id: ev_m_aside_sep
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval [%]
  value: '45.4'
  unit: percent
  direction: higher_better
  location: Table 3
  page: 23
comparisons:
- id: c_l_bipia
  left: ev_l_aside_bipia_text
  right: ev_l_van_bipia_text
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 7
- id: c_m_bipia
  left: ev_m_aside_bipia_text
  right: ev_m_van_bipia_text
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 7
- id: c_m_struq
  left: ev_m_aside_struq_id
  right: ev_m_van_struq_id
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 7
- id: c_l_sep
  left: ev_l_aside_sep
  right: ev_l_van_sep
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  page: 23
- id: c_m_sep
  left: ev_m_aside_sep
  right: ev_m_van_sep
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  page: 23
claims:
- id: k1
  text: instruction-tuning with ASIDE gives much higher instruction-data separation without performance loss
  location: abstract
  anchor: achieves substantially higher instruction-data separation without performance loss
  jobs:
  - general_utility
  - R2
  supported_by:
  - c_l_sep
  - c_m_sep
  page: 1
- id: k2
  text: ASIDE makes models more robust on prompt injection benchmarks without dedicated safety training
  location: abstract
  anchor: makes the models more robust to prompt injection benchmarks, even without dedicated safety training
  jobs:
  - R1
  supported_by:
  - c_l_bipia
  - c_m_bipia
  - c_m_struq
  page: 1
coverage:
  R1: direct
  R2clean: proxy
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
schema_gaps:
- where: defences aside_*
  note: the signal is a fixed, non-learned rotation of the data tokens' embeddings; axis 1 has learned token features but no value for a fixed embedding transform
- where: defences aside_*
  note: signal dependence was measured only on Qwen3-8B (rotation-angle ablation, Figure 7, p. 17, values in a figure); v0 cannot record a test made on another checkpoint of the same method
- where: evaluations
  note: the undefended reference is an instruction-tuned twin trained on the same data with the standard template (Vanilla), not an off-the-shelf model; v0 has only the none:<base> convention and no field for how the reference
    was built
- where: results r10, r11, r18, r19
  note: the SEP separation score mixes executing the probe as an instruction and not executing it as data; it maps to no single quantity, coded other
- where: results
  note: values are means over three independent runs with a standard deviation or standard error; v0 has no field for the spread or the number of runs
- where: comparisons c_l_sep, c_m_sep
  note: the decoding settings for SEP and AlpacaEval are not given on the pages read (greedy is stated for the injection benchmarks, p. 20), so c6 is unreported and the grade is U
- where: claim k1
  note: the no-loss claim rests on AlpacaEval (general utility); SEP Utility, the closer measure of doing the user's task, falls for Mistral 7B (45.0 to 25.3) and Llama 3.1 8B (70.3 to 63.0), which the paper attributes to
    brittle evaluation (p. 5); v0 has no field to mark a claim as contradicted by a result in the same report

In [ ]:
%%writefile reports/pilot_v0/countersteer_arxiv2026.yaml
report:
  id: countersteer_arxiv2026
  slug: '2609.36570'
  title: 'CounterSteer: Suppressing Indirect Prompt Injection with Activation Steering'
  first_author: Mark Russinovich
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2609.36570'
  version_read: arXiv v1
  first_date: '2026-09-29'
  read: F
  role: mixed
  code_release: manifest of the AgentDojo grid stated as released; code not checked in this pass
defences:
- id: cs_gptoss20b
  name: CounterSteer
  origin: this_report
  base_checkpoint: gpt-oss-20b
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  components:
  - per-model residual-stream direction fitted from paired episodes that differ only in whether an embedded instruction is followed
  - the direction subtracted at every tool-result token during prefill
  isolation: ablation_present
  layout: agent transcript; the edit applies to the tool-result spans, which the serving stack must mark
  signal_dependence: stated
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  released_weights: not applicable (weights unchanged)
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  anchor: only white-box serving and tool-result span boundaries
- id: cs_qwen3_30b
  name: CounterSteer
  origin: this_report
  base_checkpoint: Qwen3-30B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  layout: as cs_gptoss20b
  signal_dependence: stated
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  released_weights: not applicable (weights unchanged)
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
- id: cs_llama31_8b
  name: CounterSteer
  origin: this_report
  base_checkpoint: Llama-3.1-8B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  layout: as cs_gptoss20b
  signal_dependence: stated
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  released_weights: not applicable (weights unchanged)
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
- id: spot_gptoss20b
  name: Spotlighting (re-measured by CounterSteer)
  origin: other_report
  base_checkpoint: gpt-oss-20b
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  layout: AgentDojo's spotlighting configuration
  signal_dependence: not_reported
  released_weights: not applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 8
evaluations:
- id: ev_g_none_ad
  defence_id: none:gpt-oss-20b
  benchmark: AgentDojo, fixed 180-episode grid over all four suites
  split: 180 of 629 user-by-injection pairs, pinned
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  sample_size: '180'
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 6
- id: ev_g_cs_ad
  defence_id: cs_gptoss20b
  benchmark: AgentDojo, fixed 180-episode grid over all four suites
  split: 180 of 629 user-by-injection pairs, pinned
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  sample_size: '180'
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 6
- id: ev_q_none_ad
  defence_id: none:Qwen3-30B
  benchmark: AgentDojo, fixed 180-episode grid over all four suites
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  sample_size: '180'
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 6
- id: ev_q_cs_ad
  defence_id: cs_qwen3_30b
  benchmark: AgentDojo, fixed 180-episode grid over all four suites
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  sample_size: '180'
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 6
- id: ev_l_none_ad
  defence_id: none:Llama-3.1-8B
  benchmark: AgentDojo, fixed 180-episode grid over all four suites
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  sample_size: '180'
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 6
- id: ev_l_cs_ad
  defence_id: cs_llama31_8b
  benchmark: AgentDojo, fixed 180-episode grid over all four suites
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  sample_size: '180'
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 6
- id: ev_g_none_batt
  defence_id: none:gpt-oss-20b
  benchmark: AgentDojo, fixed 180-episode grid (comparison battery run)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 8
- id: ev_g_spot_batt
  defence_id: spot_gptoss20b
  benchmark: AgentDojo, fixed 180-episode grid (comparison battery run)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 8
- id: ev_g_cs_batt
  defence_id: cs_gptoss20b
  benchmark: AgentDojo, fixed 180-episode grid (comparison battery run)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo's own security and utility checkers
  denominator: 180 episodes
  decoding: greedy, 4096-token per-turn budget
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 8
- id: ev_g_none_autodojo
  defence_id: none:gpt-oss-20b
  benchmark: AutoDojo over AgentDojo, 36 cells
  attacker_type: adaptive
  attacker_access: black_box
  attacker_budget: six optimiser iterations (AD@6)
  judge: AgentDojo's end-to-end checker
  decoding: greedy, 4096-token per-turn budget
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 8
- id: ev_g_cs_autodojo
  defence_id: cs_gptoss20b
  benchmark: AutoDojo over AgentDojo, 36 cells
  attacker_type: adaptive
  attacker_access: black_box
  attacker_budget: six optimiser iterations (AD@6)
  judge: AgentDojo's end-to-end checker
  decoding: greedy, 4096-token per-turn budget
  layout_same_as_defence: 'yes'
  measured_by: countersteer_arxiv2026
  page: 8
results:
- id: r1
  evaluation_id: ev_g_none_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo compromise rate (undefended)
  value: '.475'
  unit: fraction
  direction: lower_better
  location: Table II
  page: 6
- id: r2
  evaluation_id: ev_g_cs_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo compromise rate (defended)
  value: '.079'
  unit: fraction
  direction: lower_better
  location: Table II
  page: 6
- id: r3
  evaluation_id: ev_g_cs_ad
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility, % of clean (typography-normalized)
  value: '94.4'
  unit: percent
  direction: higher_better
  location: Table II
  page: 6
- id: r4
  evaluation_id: ev_q_none_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo compromise rate (undefended)
  value: '.489'
  unit: fraction
  direction: lower_better
  location: Table II
  page: 6
- id: r5
  evaluation_id: ev_q_cs_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo compromise rate (defended)
  value: '.072'
  unit: fraction
  direction: lower_better
  location: Table II
  page: 6
- id: r6
  evaluation_id: ev_q_cs_ad
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility, % of clean
  value: '94.1'
  unit: percent
  direction: higher_better
  location: Table II
  page: 6
- id: r7
  evaluation_id: ev_l_none_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo compromise rate (undefended)
  value: '.100'
  unit: fraction
  direction: lower_better
  location: Table II
  page: 6
- id: r8
  evaluation_id: ev_l_cs_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo compromise rate (defended)
  value: '.050'
  unit: fraction
  direction: lower_better
  location: Table II
  page: 6
- id: r9
  evaluation_id: ev_l_cs_ad
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility, % of clean
  value: '100.0'
  unit: percent
  direction: higher_better
  location: Table II
  page: 6
- id: r10
  evaluation_id: ev_g_none_batt
  quantity: R1
  coverage: direct
  metric_as_printed: cmp (no defense)
  value: '.483'
  unit: fraction
  direction: lower_better
  location: Table III
  page: 8
- id: r11
  evaluation_id: ev_g_none_batt
  quantity: R2clean
  coverage: direct
  metric_as_printed: 'util (no defense: absolute clean completion)'
  value: '98.1'
  unit: percent
  direction: higher_better
  location: Table III
  page: 8
- id: r12
  evaluation_id: ev_g_spot_batt
  quantity: R1
  coverage: direct
  metric_as_printed: cmp (spotlighting)
  value: '.420'
  unit: fraction
  direction: lower_better
  location: Table III
  page: 8
- id: r13
  evaluation_id: ev_g_spot_batt
  quantity: R2clean
  coverage: direct
  metric_as_printed: util (% of clean)
  value: '92.5'
  unit: percent
  direction: higher_better
  location: Table III
  page: 8
- id: r14
  evaluation_id: ev_g_cs_batt
  quantity: R1
  coverage: direct
  metric_as_printed: cmp (CounterSteer)
  value: '.091'
  unit: fraction
  direction: lower_better
  location: Table III
  page: 8
- id: r15
  evaluation_id: ev_g_cs_batt
  quantity: R2clean
  coverage: direct
  metric_as_printed: util (% of clean)
  value: '90.6'
  unit: percent
  direction: higher_better
  location: Table III
  page: 8
- id: r16
  evaluation_id: ev_g_none_autodojo
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo AD@6 (no defense)
  value: '.673'
  unit: fraction
  direction: lower_better
  location: Table III, bottom block
  page: 8
- id: r17
  evaluation_id: ev_g_cs_autodojo
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo AD@6 (CounterSteer)
  value: '.175'
  unit: fraction
  direction: lower_better
  location: Table III, bottom block
  page: 8
comparisons:
- id: c_g_ad
  left: ev_g_cs_ad
  right: ev_g_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 6
- id: c_q_ad
  left: ev_q_cs_ad
  right: ev_q_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 6
- id: c_l_ad
  left: ev_l_cs_ad
  right: ev_l_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 6
- id: c_g_spot
  left: ev_g_cs_batt
  right: ev_g_spot_batt
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 8
- id: c_g_adaptive
  left: ev_g_cs_autodojo
  right: ev_g_none_autodojo
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 8
claims:
- id: k1
  text: AgentDojo compromise falls from 0.10-0.49 to 0.006-0.079 at 93-100% normalized benign utility across five models
  location: abstract
  anchor: AgentDojo compromise rate from 0.10–0.49 to 0.006–0.079
  jobs:
  - R1
  - R2
  supported_by:
  - c_g_ad
  - c_q_ad
  - c_l_ad
  page: 1
- id: k2
  text: a benchmark-level adaptive attacker is held to about a quarter of its undefended success on the two most deeply evaluated models
  location: abstract
  anchor: held to roughly a quarter of that on the two most deeply
  jobs:
  - R1
  supported_by:
  - c_g_adaptive
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
schema_gaps:
- where: results r3, r6, r9, r13, r15
  note: benign utility is a percentage of the same row's own clean run, while the no-defence row is absolute; v0's unit list has no relative-to-own-baseline unit
- where: evaluations ev_g_*_ad and ev_g_*_batt
  note: the same configuration has a canonical run (.475 to .079) and a comparison-battery run (.483 to .091); v0 has no field for the run or regeneration an evaluation belongs to
- where: results r1 to r17
  note: the agentic compromise rates are stated to be lower bounds because of the per-turn generation budget (p. 6); v0 cannot mark a value as a bound
- where: defences cs_*
  note: the amendment-1 fields assume training; a fitted steering direction has a fitting set of paired episodes but no training targets, so the fields are coded no without a matching value
- where: report
  note: surviving compromises are split into delegated-authority tasks (the user's own prompt hands authority to retrieved content) and the rest (p. 7); v0 has no field for a task partition that changes how a compromise should
    be read
- where: coverage
  note: utility under attack is reported in the full-suite tables (cited, not coded here); v0 coverage cannot point to where in the report a quantity is found

In [ ]:
%%writefile reports/pilot_v0/jia_critical_arxiv2025.yaml
report:
  id: jia_critical_arxiv2025
  slug: '2505.18333'
  title: A Critical Evaluation of Defenses against Prompt Injection Attacks
  first_author: Yuqi Jia
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2505.18333'
  version_read: arXiv v1
  first_date: '2025-05-23'
  read: F
  role: re_measures
  code_release: https://github.com/PIEval123/PIEval
defences:
- id: secalign_llama3_8bi_rel
  name: SecAlign (checkpoint released by its authors)
  origin: other_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  layout: the released model's own delimiters; special tokens filtered from the data at run time
  signal_dependence: not_reported
  released_weights: downloaded from the SecAlign release
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  anchor: We downloaded LLMs released by StruQ and SecAlign
- id: struq_llama3_8bi_rel
  name: StruQ (checkpoint released by its authors)
  origin: other_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  layout: the released model's own delimiters; special tokens filtered from the data at run time
  signal_dependence: not_reported
  released_weights: downloaded from the StruQ release
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  anchor: We downloaded LLMs released by StruQ and SecAlign
evaluations:
- id: ev_none_opi_comb
  defence_id: none:Llama-3-8B-Instruct
  benchmark: OpenPromptInjection (seven NLP tasks)
  split: 4,900 target and injected prompt tuples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: Combined Attack
  judge: reply equivalent to the injected task's reference response (ASV)
  denominator: 4,900 tuples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 9
- id: ev_sec_opi_comb
  defence_id: secalign_llama3_8bi_rel
  benchmark: OpenPromptInjection (seven NLP tasks)
  split: 4,900 target and injected prompt tuples
  attacker_type: static
  attacker_access: black_box
  attacker_budget: Combined Attack
  judge: reply equivalent to the injected task's reference response (ASV)
  denominator: 4,900 tuples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 9
- id: ev_none_opi_gcg
  defence_id: none:Llama-3-8B-Instruct
  benchmark: OpenPromptInjection (seven NLP tasks)
  split: 50 tuples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: GCG optimising the separator
  judge: reply equivalent to the injected task's reference response (ASV)
  denominator: 50 tuples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 9
- id: ev_sec_opi_gcg
  defence_id: secalign_llama3_8bi_rel
  benchmark: OpenPromptInjection (seven NLP tasks)
  split: 50 tuples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: GCG optimising the separator
  judge: reply equivalent to the injected task's reference response (ASV)
  denominator: 50 tuples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 9
- id: ev_sec_opi_gcg_adapt
  defence_id: secalign_llama3_8bi_rel
  benchmark: OpenPromptInjection (seven NLP tasks)
  split: 50 tuples
  attacker_type: adaptive
  attacker_access: white_box
  attacker_budget: GCG with delimiters built from the nearest tokens to the defence's special tokens
  judge: reply equivalent to the injected task's reference response (ASV)
  denominator: 50 tuples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 9
- id: ev_struq_opi_gcg
  defence_id: struq_llama3_8bi_rel
  benchmark: OpenPromptInjection (seven NLP tasks)
  split: 50 tuples
  attacker_type: optimisation
  attacker_access: white_box
  attacker_budget: GCG optimising the separator
  judge: reply equivalent to the injected task's reference response (ASV)
  denominator: 50 tuples
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 9
- id: ev_sec_alpaca_rel
  defence_id: secalign_llama3_8bi_rel
  benchmark: AlpacaEval on AlpacaFarm, reference model Llama-3-8B-Instruct
  attacker_type: none
  attacker_access: not_applicable
  judge: AlpacaEval win rate against the undefended Llama-3-8B-Instruct
  decoding: not reported
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 8
- id: ev_none_opi_util
  defence_id: none:Llama-3-8B-Instruct
  benchmark: OpenPromptInjection clean prompt-response pairs
  split: 700 pairs, 100 per task
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy on the target task
  decoding: not reported
  sample_size: '700'
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 8
- id: ev_sec_opi_util
  defence_id: secalign_llama3_8bi_rel
  benchmark: OpenPromptInjection clean prompt-response pairs
  split: 700 pairs, 100 per task
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy on the target task
  decoding: not reported
  sample_size: '700'
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 8
- id: ev_none_mmlu_util
  defence_id: none:Llama-3-8B-Instruct
  benchmark: MMLU-PI clean pairs
  split: 200 MMLU test prompts
  attacker_type: none
  attacker_access: not_applicable
  judge: MMLU accuracy
  decoding: not reported
  sample_size: '200'
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 8
- id: ev_sec_mmlu_util
  defence_id: secalign_llama3_8bi_rel
  benchmark: MMLU-PI clean pairs
  split: 200 MMLU test prompts
  attacker_type: none
  attacker_access: not_applicable
  judge: MMLU accuracy
  decoding: not reported
  sample_size: '200'
  layout_same_as_defence: 'yes'
  measured_by: jia_critical_arxiv2025
  page: 8
results:
- id: r1
  evaluation_id: ev_none_opi_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASV, Combined Attack (existing)
  value: '0.52'
  unit: fraction
  direction: lower_better
  location: Table 2
  page: 9
- id: r2
  evaluation_id: ev_sec_opi_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASV, Combined Attack (existing)
  value: '0.06'
  unit: fraction
  direction: lower_better
  location: Table 2
  page: 9
- id: r3
  evaluation_id: ev_none_opi_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASV, GCG (existing)
  value: '1.00'
  unit: fraction
  direction: lower_better
  location: Table 2
  page: 9
- id: r4
  evaluation_id: ev_sec_opi_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASV, GCG (existing)
  value: '0.24'
  unit: fraction
  direction: lower_better
  location: Table 2
  page: 9
- id: r5
  evaluation_id: ev_sec_opi_gcg_adapt
  quantity: R1
  coverage: direct
  metric_as_printed: ASV, GCG (adaptive)
  value: '0.46'
  unit: fraction
  direction: lower_better
  location: Table 2
  page: 9
- id: r6
  evaluation_id: ev_struq_opi_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASV, GCG (existing)
  value: '0.80'
  unit: fraction
  direction: lower_better
  location: Table 2
  page: 9
- id: r7
  evaluation_id: ev_sec_alpaca_rel
  quantity: general_utility
  coverage: direct
  metric_as_printed: Win Rate (%) against Llama-3-8B-Instruct
  value: '36.42'
  unit: percent
  direction: higher_better
  location: Table 1(a)
  page: 8
- id: r8
  evaluation_id: ev_none_opi_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: absolute utility, OpenPromptInjection
  value: '0.65'
  unit: fraction
  direction: higher_better
  location: Table 1(b)
  page: 8
- id: r9
  evaluation_id: ev_sec_opi_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: absolute utility, OpenPromptInjection
  value: '0.48'
  unit: fraction
  direction: higher_better
  location: Table 1(b)
  page: 8
- id: r10
  evaluation_id: ev_none_mmlu_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: absolute utility, MMLU-PI
  value: '0.45'
  unit: fraction
  direction: higher_better
  location: Table 1(b)
  page: 8
- id: r11
  evaluation_id: ev_sec_mmlu_util
  quantity: general_utility
  coverage: direct
  metric_as_printed: absolute utility, MMLU-PI
  value: '0.34'
  unit: fraction
  direction: higher_better
  location: Table 1(b)
  page: 8
comparisons:
- id: c_comb
  left: ev_sec_opi_comb
  right: ev_none_opi_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 9
- id: c_gcg
  left: ev_sec_opi_gcg
  right: ev_none_opi_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 9
- id: c_adapt
  left: ev_sec_opi_gcg_adapt
  right: ev_sec_opi_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched
  page: 9
- id: c_util
  left: ev_sec_opi_util
  right: ev_none_opi_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 8
claims:
- id: k1
  text: existing defences are not as successful as previously reported when evaluated with diverse prompts, adaptive attacks and absolute utility
  location: abstract
  anchor: we show that existing defenses are not as successful as previously reported
  jobs:
  - R1
  - R2
  - general_utility
  supported_by:
  - c_gcg
  - c_adapt
  - c_util
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
schema_gaps:
- where: report
  note: only Section 5.1 (StruQ and SecAlign on Llama-3-8B-Instruct) was coded in the pilot; Instruction Hierarchy, the detectors and the Appendix base-model runs were not; v0 has no field for the extraction scope within a
    report
- where: evaluation ev_sec_alpaca_rel
  note: the win rate uses the undefended model as the reference instead of davinci003, so the number is relative to the baseline by construction; v0 has no reference-model field for a relative judge
- where: evaluations ev_*_gcg
  note: the optimisation-based runs use a 50-tuple subset of the 4,900; v0's split field is free text and cannot link a subset to its parent set
- where: comparisons
  note: the report's central point is a disagreement with SecAlign's own results (utility and GCG success); v0 has no cross-report link from an evaluation here to the original report's evaluation of the same checkpoint (secalign_ccs2025)

In [ ]:
%%writefile reports/pilot_v0/secfid_arxiv2026.yaml
report:
  id: secfid_arxiv2026
  slug: '2606.30783'
  title: 'Security–Fidelity Tradeoffs: The Hidden Cost of Prompt Injection Defense'
  first_author: Mitchell Hermon
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2606.30783'
  version_read: arXiv v1
  first_date: '2026-06-29'
  read: F
  role: mixed
  code_release: not checked in this pass
defences:
- id: aside_llama31_8b_sf
  name: ASIDE (re-measured by SecFid)
  origin: other_report
  base_checkpoint: Llama 3.1 8B
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  layout: the inference procedure specified by the defence
  signal_dependence: not_reported
  released_weights: released checkpoint used
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 24
- id: ise_llama31_8b_sf
  name: ISE (re-measured by SecFid)
  origin: other_report
  base_checkpoint: Llama 3.1 8B
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  layout: the inference procedure specified by the defence
  signal_dependence: not_reported
  released_weights: released checkpoint used
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 24
- id: metasecalign_8b_sf
  name: SecAlign as printed; the checkpoint is facebook/Meta-SecAlign-8B (LoRA adapter)
  origin: other_report
  base_checkpoint: Llama 3.1 8B with the facebook/Meta-SecAlign-8B adapter
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  layout: the inference procedure specified by the defence (data in its own input role)
  signal_dependence: not_reported
  released_weights: facebook/Meta-SecAlign-8B
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 24
  anchor: facebook/Meta-SecAlign-8B
- id: deftokens_llama31_8b_sf
  name: DefensiveTokens (re-measured by SecFid)
  origin: other_report
  base_checkpoint: Llama 3.1 8B
  axis1:
  - learned_token_features
  axis2:
  - soft_tokens_frozen
  isolation: not_applicable
  layout: five defensive tokens prepended to the input
  signal_dependence: not_reported
  released_weights: released tokens used
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 24
  anchor: DefensiveTokens is evaluated with five defensive tokens prepended to the input
- id: secfid_dpo_8b
  name: Fidelity-aware DPO (SecFid preference tuning)
  origin: this_report
  base_checkpoint: Meta-SecAlign-8B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  components:
  - Meta-SecAlign-8B starting point
  - DPO with processed preferred to ignored and ignored preferred to executed, on counting, extraction and translation
  isolation: ablation_present
  layout: as metasecalign_8b_sf
  signal_dependence: not_reported
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  released_weights: not checked in this pass
  trains_instruction_pair: 'no'
  trains_carry_through: yes_by_preference
  page: 7
  anchor: We build preferences from the three-way labels with the ordering
evaluations:
- id: ev_base
  defence_id: none:Llama 3.1 8B
  benchmark: 'SecFid (1,168 instances: counting, extraction, translation, editing)'
  attacker_type: static
  attacker_access: black_box
  attacker_budget: one probe per instance, plain or persuasive framing, prefix, inside or suffix
  judge: signature answer for execution; set, count or embedding-similarity checks for processed or ignored
  denominator: 1,168 instances
  decoding: temperature 0
  sample_size: '1168'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 6
- id: ev_aside
  defence_id: aside_llama31_8b_sf
  benchmark: SecFid (1,168 instances)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: as ev_base
  judge: as ev_base
  denominator: 1,168 instances
  decoding: temperature 0
  sample_size: '1168'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 6
- id: ev_ise
  defence_id: ise_llama31_8b_sf
  benchmark: SecFid (1,168 instances)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: as ev_base
  judge: as ev_base
  denominator: 1,168 instances
  decoding: temperature 0
  sample_size: '1168'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 6
- id: ev_msa
  defence_id: metasecalign_8b_sf
  benchmark: SecFid (1,168 instances)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: as ev_base
  judge: as ev_base
  denominator: 1,168 instances
  decoding: temperature 0
  sample_size: '1168'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 6
- id: ev_dt
  defence_id: deftokens_llama31_8b_sf
  benchmark: SecFid (1,168 instances)
  attacker_type: static
  attacker_access: black_box
  attacker_budget: as ev_base
  judge: as ev_base
  denominator: 1,168 instances
  decoding: temperature 0
  sample_size: '1168'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 6
- id: ev_msa_core
  defence_id: metasecalign_8b_sf
  benchmark: SecFid core held-out split (counting, extraction, translation)
  split: 141 locked cases
  attacker_type: static
  attacker_access: black_box
  judge: as ev_base
  denominator: '141'
  decoding: temperature 0
  sample_size: '141'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 7
- id: ev_dpo_core
  defence_id: secfid_dpo_8b
  benchmark: SecFid core held-out split (counting, extraction, translation)
  split: 141 locked cases
  attacker_type: static
  attacker_access: black_box
  judge: as ev_base
  denominator: '141'
  decoding: temperature 0
  sample_size: '141'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 7
- id: ev_msa_edit
  defence_id: metasecalign_8b_sf
  benchmark: SecFid editing task, held out from training
  split: 273 cases
  attacker_type: static
  attacker_access: black_box
  judge: as ev_base
  denominator: '273'
  decoding: temperature 0
  sample_size: '273'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 7
- id: ev_dpo_edit
  defence_id: secfid_dpo_8b
  benchmark: SecFid editing task, held out from training
  split: 273 cases
  attacker_type: static
  attacker_access: black_box
  judge: as ev_base
  denominator: '273'
  decoding: temperature 0
  sample_size: '273'
  layout_same_as_defence: 'yes'
  measured_by: secfid_arxiv2026
  page: 7
results:
- id: r_base_e
  evaluation_id: ev_base
  quantity: R1
  coverage: direct
  metric_as_printed: Exec.
  value: '49.1'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
- id: r_base_i
  evaluation_id: ev_base
  quantity: R3ns
  coverage: direct
  metric_as_printed: Ign.
  value: '7.7'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
- id: r_base_p
  evaluation_id: ev_base
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '50.2'
  unit: percent
  direction: higher_better
  location: Table 1
  page: 6
- id: r_aside_e
  evaluation_id: ev_aside
  quantity: R1
  coverage: direct
  metric_as_printed: Exec.
  value: '5.7'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
- id: r_aside_i
  evaluation_id: ev_aside
  quantity: R3ns
  coverage: direct
  metric_as_printed: Ign.
  value: '24.7'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
- id: r_aside_p
  evaluation_id: ev_aside
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '39.6'
  unit: percent
  direction: higher_better
  location: Table 1
  page: 6
- id: r_ise_e
  evaluation_id: ev_ise
  quantity: R1
  coverage: direct
  metric_as_printed: Exec.
  value: '14.0'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
- id: r_ise_i
  evaluation_id: ev_ise
  quantity: R3ns
  coverage: direct
  metric_as_printed: Ign.
  value: '20.3'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
- id: r_ise_p
  evaluation_id: ev_ise
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '53.6'
  unit: percent
  direction: higher_better
  location: Table 1
  page: 6
- id: r_msa_e
  evaluation_id: ev_msa
  quantity: R1
  coverage: direct
  metric_as_printed: Exec.
  value: '0.7'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
- id: r_msa_i
  evaluation_id: ev_msa
  quantity: R3ns
  coverage: direct
  metric_as_printed: Ign.
  value: '26.1'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
- id: r_msa_p
  evaluation_id: ev_msa
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '67.0'
  unit: percent
  direction: higher_better
  location: Table 1
  page: 6
- id: r_dt_e
  evaluation_id: ev_dt
  quantity: R1
  coverage: direct
  metric_as_printed: Exec.
  value: '1.9'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
- id: r_dt_i
  evaluation_id: ev_dt
  quantity: R3ns
  coverage: direct
  metric_as_printed: Ign.
  value: '53.7'
  unit: percent
  direction: lower_better
  location: Table 1
  page: 6
- id: r_dt_p
  evaluation_id: ev_dt
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '37.7'
  unit: percent
  direction: higher_better
  location: Table 1
  page: 6
- id: r_core_msa_i
  evaluation_id: ev_msa_core
  quantity: R3ns
  coverage: direct
  metric_as_printed: Ign.
  value: '17.7'
  unit: percent
  direction: lower_better
  location: Table 2
  page: 7
- id: r_core_msa_p
  evaluation_id: ev_msa_core
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '73.8'
  unit: percent
  direction: higher_better
  location: Table 2
  page: 7
- id: r_core_dpo_i
  evaluation_id: ev_dpo_core
  quantity: R3ns
  coverage: direct
  metric_as_printed: Ign.
  value: '9.2'
  unit: percent
  direction: lower_better
  location: Table 2
  page: 7
- id: r_core_dpo_p
  evaluation_id: ev_dpo_core
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '83.0'
  unit: percent
  direction: higher_better
  location: Table 2
  page: 7
- id: r_edit_msa_p
  evaluation_id: ev_msa_edit
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '43.2'
  unit: percent
  direction: higher_better
  location: Table 2
  page: 7
- id: r_edit_dpo_p
  evaluation_id: ev_dpo_edit
  quantity: R3cp
  coverage: direct
  metric_as_printed: Proc.
  value: '80.6'
  unit: percent
  direction: higher_better
  location: Table 2
  page: 7
- id: r_edit_dpo_e
  evaluation_id: ev_dpo_edit
  quantity: R1
  coverage: direct
  metric_as_printed: Exec.
  value: '0.7'
  unit: percent
  direction: lower_better
  location: Table 2
  page: 7
comparisons:
- id: c_aside
  left: ev_aside
  right: ev_base
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 6
- id: c_msa
  left: ev_msa
  right: ev_base
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 6
- id: c_dt
  left: ev_dt
  right: ev_base
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 6
- id: c_dpo_core
  left: ev_dpo_core
  right: ev_msa_core
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 7
- id: c_dpo_edit
  left: ev_dpo_edit
  right: ev_msa_edit
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  page: 7
claims:
- id: k1
  text: defences resist injected instructions largely by suppressing untrusted text, and no model or defence achieves both security and fidelity
  location: abstract
  anchor: defenses resist injected instructions largely by suppressing untrusted text
  jobs:
  - R1
  - R3
  supported_by:
  - c_aside
  - c_msa
  - c_dt
  page: 1
- id: k2
  text: fidelity-aware preference tuning raises processing while keeping security
  location: Section 4.3
  anchor: S EC F ID preference tuning improves processing while
  jobs:
  - R1
  - R3
  supported_by:
  - c_dpo_core
  - c_dpo_edit
  page: 7
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: proxy
  R3ns: direct
  R3cp: direct
  R3safe: direct
schema_gaps:
- where: defence metasecalign_8b_sf
  note: the paper prints SecAlign but evaluates the released facebook/Meta-SecAlign-8B adapter (p. 24); v0 has one name field and no checkpoint identifier, so the two defences can be conflated across reports
- where: results r_*_p
  note: Processed is both correct handling of the data (R3cp) and the user's task done under attack (R2attack proxy) in translation and editing; v0 allows one quantity per printed result
- where: results
  note: the base rows of Table 1 report the paired change from base in parentheses next to each defended rate; v0 has no field for a printed difference
- where: results
  note: safe processing (R3safe) is defined (p. 4) and reported by task and framing (Tables 15 to 23), not per defence in Table 1; v0 coverage cannot say that a quantity is reported only at an aggregate level
- where: claim k2
  note: the supporting split was locked before testing and one task (editing) was held out entirely; v0 has no field for whether a test split was locked or held out from training

## Stage 2. The PDFs read are the PDFs registered

In [ ]:
import csv, json, logging, datetime as dt
from collections import Counter
import schema, search, corpus, paths
for m in (schema, search, corpus, paths):
    importlib.reload(m)
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader

PILOT = ['spotlighting_arxiv2024', 'secalign_ccs2025', 'aside_iclr2026', 'countersteer_arxiv2026', 'jia_critical_arxiv2025', 'secfid_arxiv2026']
man = {m["slug"]: m for m in search.read_csv(paths.REPORTS / "defences.csv")}
problems, texts = [], {}
for rid in PILOT:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    pdf = paths.DATA_DEFENCES / f"{slug}.pdf"
    m = man.get(slug, {})
    if not pdf.exists():
        problems.append(f"{rid}: {pdf.name} not in data/defences"); continue
    sha = corpus.sha256(pdf)
    rd = PdfReader(str(pdf))
    texts[slug] = ([p.extract_text() or "" for p in rd.pages], [p.extract_text(extraction_mode="layout") or "" for p in rd.pages])
    ok_sha = sha == m.get("sha256")
    ok_inc = m.get("decision") == "include"
    print(f"{rid:<26} {slug:<12} pages {len(rd.pages):>3}  sha256 {'matches' if ok_sha else 'DIFFERS'}  decision {m.get('decision')}")
    if not ok_sha:
        problems.append(f"{rid}: PDF differs from the one registered in reports/defences.csv")
    if not ok_inc:
        problems.append(f"{rid}: not an included study")

## Stage 3. Validate the six records against the frozen schema and their PDFs; grade the comparisons

In [ ]:
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
check_rows, summary, residual = [], [], []
for rid in PILOT:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    plain, lay = texts.get(slug, (None, None))
    probs, notes = schema.validate_record(rec, S1, plain, lay)
    grades = Counter(schema.grade(c) for c in rec.get("comparisons") or [])
    n_res = len(rec.get("results") or [])
    n_val = sum(1 for r in rec.get("results") or [] if r.get("value_source") != "figure_only")
    n_anc = sum(1 for k in ("defences", "claims") for r in rec.get(k) or [] if r.get("anchor"))
    summary.append({"report": rid, "defences": len(rec.get("defences") or []), "evaluations": len(rec.get("evaluations") or []),
                    "results": n_res, "values_checked": n_val, "anchors_checked": n_anc,
                    "comparisons": len(rec.get("comparisons") or []), "grade_A": grades["A"], "grade_B": grades["B"],
                    "grade_C": grades["C"], "grade_U": grades["U"], "claims": len(rec.get("claims") or []),
                    **{f"cov_{q}": rec["coverage"][q] for q in ("R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe")},
                    "problems": len(probs)})
    check_rows += [{"report": rid, "status": "problem", "detail": p} for p in probs]
    residual += [{"report": rid, "schema": "v1", "where": n.split(": ", 1)[0], "note": n.split(": ", 1)[1]} for n in notes]
    print(f"{rid:<26} problems {len(probs)}; values on page {n_val}; anchors on page {n_anc}; grades {dict(grades)}")
    for p in probs:
        print("   ", p)
    problems += [f"{rid}: {p}" for p in probs]
search.write_csv(paths.REPORTS / "pilot_03_check.csv", check_rows or [{"report": "all", "status": "ok", "detail": "no problems"}],
                 ["report", "status", "detail"])
search.write_csv(paths.REPORTS / "pilot_03_summary.csv", summary, list(summary[0].keys()))

## Stage 4. The pilot under schema v0: every free-text workaround

In [ ]:
S0 = schema.load_schema(paths.SCHEMA / "record_schema_v0.yaml")
gaps = []
for rid in PILOT:
    rec0 = schema.load_record(paths.REPORTS / "pilot_v0" / f"{rid}.yaml")
    plain, lay = texts.get(rec0["report"]["slug"], (None, None))
    probs0, notes0 = schema.validate_record(rec0, S0, plain, lay)
    if probs0:
        problems += [f"pilot v0 {rid}: {p}" for p in probs0]
    gaps += [{"report": rid, "schema": "v0", "where": n.split(": ", 1)[0], "note": n.split(": ", 1)[1]} for n in notes0]
    print(f"{rid:<26} v0 problems {len(probs0)}; schema gaps {len(notes0)}")
search.write_csv(paths.REPORTS / "pilot_03_gaps.csv", gaps + residual, ["report", "schema", "where", "note"])
print(f"schema gaps under v0: {len(gaps)}; residual notes under v1: {len(residual)}")
for g in residual:
    print(f"  v1 residual, {g['report']}: {g['note'][:140]}")

## Stage 5. What changed from v0 to v1 (computed from the two schema files)

In [ ]:
added_fields, added_values, now_required = [], [], []
for kind, k1 in S1["kinds"].items():
    f0 = S0["kinds"].get(kind, {"fields": {}})["fields"]
    for name, spec in k1["fields"].items():
        if name not in f0:
            added_fields.append(f"{kind}.{name}")
            continue
        new = [v for v in spec.get("values", []) if v not in f0[name].get("values", [])]
        if new:
            added_values.append(f"{kind}.{name}: " + ", ".join(new))
        if spec.get("required") and not f0[name].get("required"):
            now_required.append(f"{kind}.{name}")
print("fields added:", len(added_fields)); print("  " + "; ".join(added_fields))
print("values added:"); [print("  " + v) for v in added_values]
print("newly required:", now_required or "none")

## Stage 6. Decision record and tests

In [ ]:
tot = {k: sum(s[k] for s in summary) for k in ("defences", "evaluations", "results", "values_checked", "anchors_checked",
                                              "comparisons", "grade_A", "grade_B", "grade_C", "grade_U", "claims")}
status = "SCHEMA FROZEN (v1)" if not problems else "SCHEMA NOT FROZEN: PROBLEMS BELOW"
lines = [f"decision_03: {status}", f"this step: {dt.date.today().isoformat()}",
         "pilot reports: " + ", ".join(PILOT),
         f"records: {tot['defences']} defences, {tot['evaluations']} evaluations, {tot['results']} results, "
         f"{tot['comparisons']} comparisons, {tot['claims']} claims",
         f"checked against the PDFs: {tot['values_checked']} printed values and {tot['anchors_checked']} anchors, each found on its page",
         f"comparison grades: A {tot['grade_A']}, B {tot['grade_B']}, C {tot['grade_C']}, U {tot['grade_U']}",
         f"schema gaps under v0: {len(gaps)}; answered in v1 by {len(added_fields)} new fields, {len(added_values)} widened value "
         f"sets and 2 coding rules (schema/pilot_changes.md); residual notes under v1: {len(residual)}",
         "frozen: schema/record_schema.yaml (v1) with schema/coding_rules.md",
         "next: notebook 04 (full extraction of the included studies, with their full-text check)"] + [f"- problem: {p}" for p in problems]
(paths.REPORTS / "decision_03.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 7. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "03 pilot extraction: record schema and coding rules, six reports coded from their PDFs with values and anchors checked, one revision, schema frozen"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)